# ACS Income — Data Mining Pipeline

Pipeline: **create_dataset** → **feature_importance** (CategoricalBoCSoR) → **association_rules** (FP-Growth)

**Settings richiesti:**
- Accelerator: **GPU T4**
- Internet: **On**
- Persistence: **Files only**


In [ ]:
!pip install -q folktables catboost mlxtend scikit-learn

In [ ]:
import os
for d in ["src", "data", "results"]:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")


In [ ]:
%%writefile src/create_dataset.py
"""
create_dataset.py
=================
Downloads ACS PUMS person-level microdata for US regions (Northeast, South, 
and USA globally), applies adult_filter, bins continuous variables, maps 
categorical codes to human-readable labels, and writes one CSV per region to disk.
"""

import os
import time
import threading
import numpy as np
import pandas as pd
import folktables
from folktables import ACSDataSource
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------------------------------------------------------------------------
# Hardware-aware worker counts
# ---------------------------------------------------------------------------
_CORES           = os.cpu_count() or 4
_DOWNLOAD_WORKERS = min(_CORES * 2, 16)
_REGION_WORKERS   = 3

# ---------------------------------------------------------------------------
# Geographic scope
# DC is excluded: not present in the folktables state list.
# ---------------------------------------------------------------------------
NORTHEAST_STATES = ['CT', 'ME', 'MA', 'NH', 'RI', 'VT', 'NJ', 'NY', 'PA']
SOUTH_STATES     = [
    'DE', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'WV',
    'AL', 'KY', 'MS', 'TN', 'AR', 'LA', 'OK', 'TX',
]
USA_STATES = [
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY'
]

# ---------------------------------------------------------------------------
# ACS PUMS feature set used for tasks
# ---------------------------------------------------------------------------
_INCOME_FEATURES = [
    'AGEP',     # age
    'COW',      # class of worker
    'SCHL',     # educational attainment
    'MAR',      # marital status
    'OCCP',     # occupation
    'POBP',     # place of birth
    'RELSHIPP', # relationship to reference person
    'WKHP',     # usual hours worked per week past 12 months
    'SEX',      # sex
    'RAC1P',    # race
    'ST',       # state of current residence (Spatial Feature)
]

# ---------------------------------------------------------------------------
# Continuous → categorical binning
# ---------------------------------------------------------------------------
AGEP_BINS   = [0, 24, 34, 44, 54, 64, 200]
AGEP_LABELS = [
    'Young',           # 16–24
    'Young-Adult',     # 25–34
    'Mid-Career',      # 35–44
    'Experienced',     # 45–54
    'Late-Career',     # 55–64
    'Retirement-Age',  # 65+
]

WKHP_BINS   = [0, 19, 34, 39, 40, 49, 200]
WKHP_LABELS = [
    'Part-Time-Low',    #  1–19 hrs/wk
    'Part-Time',        # 20–34 hrs/wk
    'Near-Full-Time',   # 35–39 hrs/wk
    'Full-Time',        # 40    hrs/wk
    'Over-Full-Time',   # 41–49 hrs/wk
    'Extended-Hours',   # 50–99 hrs/wk
]

# ---------------------------------------------------------------------------
# Categorical mappings
# ---------------------------------------------------------------------------
COW_MAP = {
    '1': 'Employee-Private-For-Profit', '2': 'Employee-Private-Non-Profit',
    '3': 'Local-Government-Employee', '4': 'State-Government-Employee',
    '5': 'Federal-Government-Employee', '6': 'Self-Employed-Not-Incorporated',
    '7': 'Self-Employed-Incorporated', '8': 'Unpaid-Family-Worker',
    '9': 'Unemployed-5plus-Years-Or-Never-Worked',
}

SCHL_MAP = {
    '1': 'No-Schooling-Completed', '2': 'Nursery-School-Preschool',
    '3': 'Kindergarten', '4': 'Grade-1', '5': 'Grade-2', '6': 'Grade-3',
    '7': 'Grade-4', '8': 'Grade-5', '9': 'Grade-6', '10': 'Grade-7',
    '11': 'Grade-8', '12': 'Grade-9', '13': 'Grade-10', '14': 'Grade-11',
    '15': 'Grade-12-No-Diploma', '16': 'Regular-HS-Diploma',
    '17': 'GED-Or-Alt-Credential', '18': 'Some-College-Less-Than-1yr',
    '19': 'Some-College-1yr-Or-More-No-Degree', '20': 'Associates-Degree',
    '21': 'Bachelors-Degree', '22': 'Masters-Degree',
    '23': 'Professional-Degree-Beyond-Bachelors', '24': 'Doctorate-Degree',
}

MAR_MAP = {
    '1': 'Married', '2': 'Widowed', '3': 'Divorced',
    '4': 'Separated', '5': 'Never-Married-Or-Under-15',
}

RELSHIPP_MAP = {
    '20': 'Reference-Person', '21': 'Opposite-Sex-Husband-Wife-Spouse',
    '22': 'Opposite-Sex-Unmarried-Partner', '23': 'Same-Sex-Husband-Wife-Spouse',
    '24': 'Same-Sex-Unmarried-Partner', '25': 'Biological-Son-Or-Daughter',
    '26': 'Adopted-Son-Or-Daughter', '27': 'Stepson-Or-Stepdaughter',
    '28': 'Brother-Or-Sister', '29': 'Father-Or-Mother', '30': 'Grandchild',
    '31': 'Parent-In-Law', '32': 'Son-In-Law-Or-Daughter-In-Law',
    '33': 'Other-Relative', '34': 'Roommate-Or-Housemate', '35': 'Foster-Child',
    '36': 'Other-Nonrelative', '37': 'Institutionalized-Group-Quarters',
    '38': 'Noninstitutionalized-Group-Quarters',
}

SEX_MAP = {'1': 'Male', '2': 'Female'}

RAC1P_MAP = {
    '1': 'White-Alone', '2': 'Black-Or-African-American-Alone',
    '3': 'American-Indian-Alone', '4': 'Alaska-Native-Alone',
    '5': 'American-Indian-And-Alaska-Native-Tribes', '6': 'Asian-Alone',
    '7': 'Native-Hawaiian-And-Other-Pacific-Islander-Alone',
    '8': 'Some-Other-Race-Alone', '9': 'Two-Or-More-Races',
}

POBP_MAP = {
    '1': 'Alabama', '2': 'Alaska', '4': 'Arizona', '5': 'Arkansas', '6': 'California',
    '8': 'Colorado', '9': 'Connecticut', '10': 'Delaware', '11': 'DC', '12': 'Florida',
    '13': 'Georgia', '15': 'Hawaii', '16': 'Idaho', '17': 'Illinois', '18': 'Indiana',
    '19': 'Iowa', '20': 'Kansas', '21': 'Kentucky', '22': 'Louisiana', '23': 'Maine',
    '24': 'Maryland', '25': 'Massachusetts', '26': 'Michigan', '27': 'Minnesota',
    '28': 'Mississippi', '29': 'Missouri', '30': 'Montana', '31': 'Nebraska',
    '32': 'Nevada', '33': 'New-Hampshire', '34': 'New-Jersey', '35': 'New-Mexico',
    '36': 'New-York', '37': 'North-Carolina', '38': 'North-Dakota', '39': 'Ohio',
    '40': 'Oklahoma', '41': 'Oregon', '42': 'Pennsylvania', '44': 'Rhode-Island',
    '45': 'South-Carolina', '46': 'South-Dakota', '47': 'Tennessee', '48': 'Texas',
    '49': 'Utah', '50': 'Vermont', '51': 'Virginia', '53': 'Washington',
    '54': 'West-Virginia', '55': 'Wisconsin', '56': 'Wyoming', '72': 'Puerto-Rico',
    '100': 'Born-Abroad-US-Parents', '301': 'Cuba', '302': 'Jamaica',
    '303': 'Dominican-Republic', '308': 'Haiti', '313': 'Other-Caribbean',
    '400': 'Mexico', '414': 'Guatemala', '416': 'Honduras', '417': 'El-Salvador',
    '422': 'Nicaragua', '423': 'Panama', '424': 'Other-Central-America',
    '501': 'Colombia', '507': 'Peru', '508': 'Brazil', '516': 'Venezuela',
    '523': 'Other-South-America', '600': 'Armenia', '601': 'China', '603': 'India',
    '607': 'Japan', '613': 'Philippines', '615': 'South-Korea', '618': 'Vietnam',
    '619': 'Other-Southeast-Asia', '620': 'Other-Asia', '700': 'United-Kingdom',
    '703': 'Germany', '706': 'Greece', '708': 'Ireland', '710': 'Italy',
    '714': 'Poland', '716': 'Portugal', '720': 'Russia', '724': 'Ukraine',
    '730': 'Other-Europe', '800': 'Nigeria', '803': 'Ethiopia', '804': 'Egypt',
    '820': 'Other-Africa', '900': 'Canada', '999': 'Other-NEC',
}

OCCP_MAP = {
    '0': 'Not-In-Labor-Force-Or-Under-16', '10': 'Chief-Executives',
    '20': 'General-Operations-Managers', '120': 'Financial-Managers',
    '136': 'HR-Managers', '220': 'Advertising-And-Marketing-Managers',
    '300': 'Purchasing-Managers', '310': 'Transportation-Managers',
    '330': 'Food-Service-Managers', '410': 'Medical-And-Health-Services-Managers',
    '430': 'Construction-Managers', '440': 'Other-Managers',
    '500': 'Agents-Of-Performing-Arts', '510': 'Compliance-Officers',
    '520': 'Cost-Estimators', '530': 'Human-Resources-Workers',
    '540': 'Training-And-Development-Specialists', '565': 'Logisticians',
    '600': 'Accountants-And-Auditors', '630': 'Budget-Analysts',
    '640': 'Credit-Analysts', '650': 'Financial-Analysts',
    '700': 'Management-Analysts', '726': 'Market-Research-Analysts',
    '740': 'Business-Operations-Specialists', '800': 'Buyers-And-Purchasing-Agents',
    '840': 'Claims-Adjusters', '1005': 'Computer-And-Info-Research-Scientists',
    '1006': 'Computer-Systems-Analysts', '1007': 'Information-Security-Analysts',
    '1010': 'Computer-Programmers', '1021': 'Software-Developers',
    '1022': 'Software-Quality-Assurance', '1031': 'Web-Developers',
    '1032': 'Web-And-Digital-Interface-Designers', '1050': 'Database-Administrators',
    '1065': 'Network-And-Computer-Systems-Admins', '1100': 'Computer-Support-Specialists',
    '1200': 'Actuaries', '1220': 'Operations-Research-Analysts',
    '1230': 'Statisticians', '1240': 'Data-Scientists', '2100': 'Lawyers',
    '2105': 'Judicial-Law-Clerks', '2110': 'Judges-And-Magistrates',
    '2310': 'Elementary-School-Teachers', '2320': 'Middle-School-Teachers',
    '2330': 'Secondary-School-Teachers', '2540': 'Special-Education-Teachers',
    '2550': 'Other-Teachers', '2560': 'Tutors-And-Instructors',
    '2630': 'Postsecondary-Teachers', '2640': 'Preschool-And-Kindergarten-Teachers',
    '2720': 'Art-Directors', '2740': 'Graphic-Designers',
    '2750': 'Interior-Designers', '3010': 'Chiropractors',
    '3050': 'Dietitians-And-Nutritionists', '3090': 'Emergency-Medical-Technicians',
    '3100': 'Exercise-Physiologists', '3130': 'Pharmacists',
    '3160': 'Physical-Therapists', '3230': 'Physicians-And-Surgeons',
    '3250': 'Registered-Nurses', '3255': 'Nurse-Practitioners',
    '3260': 'Occupational-Therapists', '3300': 'Dentists',
    '3420': 'Dental-Assistants', '3500': 'Licensed-Practical-Nurses',
    '3600': 'Medical-Assistants', '4000': 'Cooks-Restaurant',
    '4020': 'Food-Preparation-Workers', '4040': 'Bartenders',
    '4055': 'Fast-Food-Workers', '4110': 'Waiters-And-Waitresses',
    '4120': 'Dining-Room-Attendants', '4140': 'Dishwashers',
    '4220': 'Janitors-And-Cleaners', '4230': 'Maids-And-Housekeeping',
    '4700': 'First-Line-Retail-Supervisors', '4720': 'Cashiers',
    '4740': 'Counter-And-Rental-Clerks', '4760': 'Retail-Salespersons',
    '4800': 'Insurance-Sales-Agents', '4810': 'Securities-And-Financial-Sales',
    '4820': 'Real-Estate-Brokers-And-Agents', '4840': 'Telemarketers',
    '4850': 'Sales-Representatives', '5000': 'First-Line-Office-Supervisors',
    '5110': 'Receptionists', '5120': 'Information-Clerks',
    '5160': 'Customer-Service-Representatives', '5230': 'Payroll-And-Timekeeping-Clerks',
    '5240': 'Human-Resources-Assistants', '5260': 'Eligibility-Interviewers',
    '5420': 'Postal-Service-Workers', '5600': 'Production-Planning-Clerks',
    '5700': 'Secretaries-And-Admin-Assistants', '5820': 'Data-Entry-Keyers',
    '9600': 'Cleaners-Of-Vehicles-And-Equipment', '9620': 'Laborers-And-Material-Movers',
    '9800': 'Military-Officer-Special-Operations', '9810': 'Military-First-Line-Supervisors',
    '9820': 'Military-Enlisted-Tactical-Operations', '9830': 'Military-Rank-Not-Specified',
}

_COLUMN_FALLBACKS = {'OCCP': 'Other-Occupation', 'POBP': 'Other-NEC'}
_COLUMN_MAPS = {
    'COW': COW_MAP, 'SCHL': SCHL_MAP, 'MAR': MAR_MAP,
    'RELSHIPP': RELSHIPP_MAP, 'SEX': SEX_MAP, 'RAC1P': RAC1P_MAP,
    'POBP': POBP_MAP, 'OCCP': OCCP_MAP,
    # 'ST' (Stato) è gestito dinamicamente bypassando i codici FIPS
}

# ---------------------------------------------------------------------------
# Vectorised lookup arrays
# ---------------------------------------------------------------------------
def _build_lookup(mapping: dict, fallback: str) -> np.ndarray:
    max_code = max(int(k) for k in mapping) if mapping else 0
    arr = np.full(max_code + 2, fill_value=fallback, dtype=object)
    for k, label in mapping.items():
        arr[int(k) + 1] = label
    return arr

_LOOKUPS: dict[str, np.ndarray] = {
    col: _build_lookup(m, _COLUMN_FALLBACKS.get(col, 'Unknown'))
    for col, m in _COLUMN_MAPS.items()
}

# ---------------------------------------------------------------------------
# Transformation helpers
# ---------------------------------------------------------------------------
def _decode_column(series: pd.Series, lookup: np.ndarray) -> np.ndarray:
    codes = pd.to_numeric(series, errors='coerce').fillna(-1).to_numpy(dtype=np.int32)
    idx   = np.clip(codes + 1, 0, len(lookup) - 1)
    return lookup[idx]

def apply_categorical_mappings(df: pd.DataFrame) -> None:
    for col, lookup in _LOOKUPS.items():
        if col in df.columns:
            df[col] = pd.Categorical(_decode_column(df[col], lookup))

def _make_income_task(threshold: int) -> folktables.BasicProblem:
    return folktables.BasicProblem(
        features         = _INCOME_FEATURES,
        target           = 'PINCP',
        target_transform = lambda x: x > threshold,
        group            = 'RAC1P',
        preprocess       = folktables.adult_filter,
        # FIX: np.where(pd.isna) funziona perfettamente con NumPy array e colonne testuali
        postprocess      = lambda x: np.where(pd.isna(x), -1, x),
    )

# ---------------------------------------------------------------------------
# Download helpers
# ---------------------------------------------------------------------------
def _download_state(data_source: ACSDataSource, state: str) -> pd.DataFrame:
    df = data_source.get_data(states=[state], download=True)
    # INIEZIONE DIRETTA DELLO STATO: inseriamo la sigla (es. 'CA')
    df['ST'] = state 
    return df

# Lock that serialises the initial ACS zip download so concurrent threads
# (state-level or region-level) never corrupt the shared cache file.
_zip_lock = threading.Lock()

def parallel_get_data(data_source: ACSDataSource, states: list[str], label: str) -> pd.DataFrame:
    n_workers = min(len(states), _DOWNLOAD_WORKERS)
    print(f'  > [{label}] {len(states)} states — {n_workers} workers')

    # Download the first state under a lock so the ACS zip archive is
    # fully cached before any parallel worker tries to read it.
    with _zip_lock:
        first_df = _download_state(data_source, states[0])
    print(f'    - {states[0]} done (cache primed)')

    results: dict[str, pd.DataFrame] = {states[0]: first_df}
    remaining = states[1:]

    if remaining:
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            futures = {pool.submit(_download_state, data_source, s): s for s in remaining}
            for fut in as_completed(futures):
                state = futures[fut]
                try:
                    results[state] = fut.result()
                    print(f'    - {state} done')
                except Exception as exc:
                    raise RuntimeError(
                        f'Download failed for {state}: {exc}\n'
                        f'If the requested survey year is not yet available, try an earlier year.'
                    ) from exc

    return pd.concat([results[s] for s in states], ignore_index=True)

def build_dataset(task: folktables.BasicProblem, states: list[str], data_source: ACSDataSource, label: str) -> pd.DataFrame:
    t0 = time.perf_counter()
    print(f'\n{"=" * 70}\n{label}\n{"=" * 70}')
    raw = parallel_get_data(data_source, states, label)
    print(f'  > raw rows: {len(raw):,}  ({time.perf_counter() - t0:.1f}s)')

    features_df, labels, _ = task.df_to_pandas(raw)
    features_df['INCOME_ABOVE_THRESHOLD'] = labels.to_numpy(dtype=np.int8)

    apply_categorical_mappings(features_df)

    # Convertiamo la colonna Stato iniettata in pd.Categorical
    if 'ST' in features_df.columns:
        features_df['ST'] = pd.Categorical(features_df['ST'])

    if 'AGEP' in features_df.columns:
        features_df['AGEP'] = pd.Categorical(pd.cut(
            pd.to_numeric(features_df['AGEP'], errors='coerce'),
            bins=AGEP_BINS, labels=AGEP_LABELS, right=True,
        ))
    if 'WKHP' in features_df.columns:
        features_df['WKHP'] = pd.Categorical(pd.cut(
            pd.to_numeric(features_df['WKHP'], errors='coerce'),
            bins=WKHP_BINS, labels=WKHP_LABELS, right=True,
        ))

    pos_pct = features_df['INCOME_ABOVE_THRESHOLD'].mean() * 100
    print(f'  > filtered rows: {len(features_df):,}  |  positive class: {pos_pct:.1f}%  ({time.perf_counter() - t0:.1f}s)')
    return features_df

# ---------------------------------------------------------------------------
# Sampling
# ---------------------------------------------------------------------------
def undersample_to(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df
    target_col = 'INCOME_ABOVE_THRESHOLD'
    groups     = [g for _, g in df.groupby(target_col, observed=False)]
    per_class  = [max(1, round(len(g) / len(df) * n)) for g in groups]
    per_class[0] += n - sum(per_class)
    sampled = pd.concat([g.sample(k, random_state=seed) for g, k in zip(groups, per_class)], ignore_index=True)
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
def main(
    survey_year: str                 = '2024',
    horizon: str                     = '1-Year',
    random_seed: int                 = 42,
    output_dir: str                  = 'data',
    income_threshold_northeast: int  = 110_000,
    income_threshold_south: int      = 90_000,
    income_threshold_usa: int        = 100_000,
    regions_to_build: list[str]      = None,
) -> None:
    t0 = time.perf_counter()
    os.makedirs(output_dir, exist_ok=True)
    data_source = ACSDataSource(survey_year=survey_year, horizon=horizon, survey='person')

    if regions_to_build is None:
        regions_to_build = ['northeast', 'south']

    print(f'\n{"=" * 70}')
    print(f'ACS INCOME DATASET — {survey_year}')
    print(f'{"=" * 70}')

    region_configs = {
        'northeast': {'states': NORTHEAST_STATES, 'threshold': income_threshold_northeast},
        'south':     {'states': SOUTH_STATES,     'threshold': income_threshold_south},
        'usa':       {'states': USA_STATES,       'threshold': income_threshold_usa},
    }

    datasets = {}

    with ThreadPoolExecutor(max_workers=_REGION_WORKERS) as pool:
        futures = {}
        for reg in regions_to_build:
            cfg = region_configs[reg]
            task = _make_income_task(cfg['threshold'])
            label = f"{reg.capitalize()} (income > ${cfg['threshold']:,})"
            futures[reg] = pool.submit(build_dataset, task, cfg['states'], data_source, label)

        for reg in regions_to_build:
            datasets[reg] = futures[reg].result()

    if 'northeast' in datasets and 'south' in datasets:
        print('\n  > Undersampling South to match Northeast size...')
        datasets['south'] = undersample_to(datasets['south'], len(datasets['northeast']), seed=random_seed)

    print('\n  > Saving datasets...')
    with ThreadPoolExecutor(max_workers=min(len(datasets), 4)) as pool:
        write_futs = []
        for reg, df in datasets.items():
            
            # INIEZIONE ANNO: Inseriamo l'anno come colonna index 0 per analisi longitudinali
            df.insert(0, 'YEAR', str(survey_year))
            
            out_path = os.path.join(output_dir, f'acs_income_{reg}_{survey_year}.csv')
            pos_pct = df['INCOME_ABOVE_THRESHOLD'].mean() * 100
            print(f'  > {reg.capitalize():<10}: {len(df):>8,} rows  |  positive class: {pos_pct:.1f}%')
            write_futs.append(pool.submit(df.to_csv, out_path, index=False))
        
        for w in write_futs:
            w.result()

    print(f'\n{"=" * 70}')
    print(f'Completed in {time.perf_counter() - t0:.1f}s')
    print(f'{"=" * 70}')

if __name__ == '__main__':
    main()


In [ ]:
%%writefile src/feature_importance.py
"""
feature_importance.py
=====================
Identifies boundary-crossing feature drivers via a categorical adaptation of
BoCSoR (Boundary Crossing Solo Ratio) built on top of CatBoost and BallTree
nearest-neighbour search.

The algorithm trains a CatBoost classifier, identifies instances near the
decision boundary (Hamming-distance percentile filter), queries k opposite-
class neighbours for each boundary instance, and records which feature values
cause the model prediction to flip — producing a transaction table suitable
for FP-Growth association-rule mining.

Public API
----------
run_for_k_values(k_values, data_path, output_base_dir,
                 target_col, perc_threshold)
    Train once, run counterfactual extraction for each k, and save results.

CategoricalBoCSoR
    Class implementing fit() and explain().
"""

import ast
import io
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.neighbors import BallTree
from joblib import Parallel, delayed


# ---------------------------------------------------------------------------
# GPU detection
# ---------------------------------------------------------------------------

def _catboost_task_type() -> str:
    """
    Detect CUDA-capable GPU availability and return the appropriate CatBoost
    task_type string ('GPU' or 'CPU').
    """
    try:
        subprocess.run(
            ['nvidia-smi'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True,
        )
        print("  > GPU detected — CatBoost will use task_type='GPU'")
        return 'GPU'
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("  > No CUDA GPU detected — CatBoost will use task_type='CPU'")
        return 'CPU'


# ---------------------------------------------------------------------------
# CategoricalBoCSoR
# ---------------------------------------------------------------------------

class CategoricalBoCSoR:
    """
    Categorical adaptation of BoCSoR (Boundary Crossing Solo Ratio).

    Differences from the original paper (all deliberate):

    1. Hamming distance instead of Euclidean
       Appropriate for categorical features where no ordinal relationship
       between values exists.

    2. No midpoint interpolation
       Midpoints between two categorical instances are not meaningful.
       Following the authors' suggestion, only real instances from the
       opposite class are used as counterfactuals.  Every candidate CF is
       verified against the model before use (see _process_single_sample).

    3. Inverted swap direction
       Instead of injecting the original feature value into the CF, the CF
       feature value is injected into the original instance.  If this causes
       the prediction to switch to the CF class, the feature with that CF
       value is recorded as a driver.  This direction is more informative
       for association-rule mining because the stored itemset contains the
       actual CF values that trigger the switch, not just the feature names.

    4. All k neighbours considered
       All k nearest opposite-class neighbours are retained.  Each produces
       a separate transaction row, enabling FP-Growth to detect patterns
       across different CF contexts for the same boundary instance.

    Performance design
    ------------------
    - model.predict calls reduced from O(1 + 2k) to O(3) per boundary sample:
        * 1 call for the original instance
        * 1 batched call for all k CF verifications
        * 1 batched call for all perturbations across all valid CFs
    - Perturbation matrix built via NumPy broadcasting (no Python inner loops).
    - Per-class global indices pre-cached in fit() so workers never recompute
      np.where(y_enc == label) on every parallel call.
    - X_enc stored as C-contiguous int32 for optimal cache behaviour.
    - joblib 'loky' backend provides true multiprocessing, bypassing the GIL.
    """

    def __init__(self, k_neighbors: int = 10, perc_threshold: int = 10) -> None:
        self.k_neighbors     = k_neighbors
        self.perc_threshold  = perc_threshold
        self.model           = None
        self.feature_encoder = OrdinalEncoder(dtype=int)
        self.label_encoder   = LabelEncoder()
        self.trees: dict     = {}   # one BallTree (Hamming metric) per class
        self._task_type      = _catboost_task_type()

    # ------------------------------------------------------------------
    # fit
    # ------------------------------------------------------------------

    def fit(self, X: pd.DataFrame, y: pd.Series) -> None:
        """
        Train CatBoost on (X, y) and build one BallTree per class.

        Parameters
        ----------
        X : feature matrix (categorical and/or integer columns)
        y : binary target (0/1)
        """
        print('  > Training CatBoost and building BallTrees...')
        self.feature_names = X.columns.tolist()

        y_enc = self.label_encoder.fit_transform(y)

        # C-contiguous int32: faster NumPy slicing and better cache locality
        # when building perturbation matrices in worker processes.
        X_enc = np.ascontiguousarray(
            self.feature_encoder.fit_transform(X), dtype=np.int32
        )

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_enc, y_enc, test_size=0.2, random_state=42, stratify=y_enc
        )

        self.model = CatBoostClassifier(
            iterations=500, depth=8, learning_rate=0.05,
            verbose=50, allow_writing_files=False,
            task_type=self._task_type,
        )
        self.model.fit(
            X_tr, y_tr,
            cat_features=list(range(X_enc.shape[1])),
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
        )

        # BallTrees built on the full encoded training set so the boundary
        # percentile reflects the complete training distribution.
        for label in np.unique(y_enc):
            idx = np.where(y_enc == label)[0]
            self.trees[label] = BallTree(X_enc[idx], metric='hamming')

        self.X_enc = X_enc
        self.y_enc = y_enc

        # Pre-cache per-class global indices so worker processes never
        # recompute np.where(y_enc == label) on every parallel call.
        self.class_indices = {
            int(label): np.where(y_enc == label)[0]
            for label in np.unique(y_enc)
        }

    # ------------------------------------------------------------------
    # _process_single_sample  (worker dispatched by joblib)
    # ------------------------------------------------------------------

    def _process_single_sample(
        self,
        sample_idx: int,
        orig_enc_sample: np.ndarray,
        ind_row: np.ndarray,
        opp_label: int,
    ):
        """
        Process one boundary instance against its k CF neighbours.

        Performance profile per call:
            model.predict calls : 3  (vs. 1 + 2k in the naive version)
            NumPy ops           : fully vectorised, no Python inner loops

        Steps
        -----
        1.  Predict orig_pred with a single call.
        2.  Retrieve all k CF candidates; verify all of them in one batched
            predict call; discard those not predicted as opp_label.
        3.  Build the full perturbation matrix in one NumPy operation:
                shape = (total_diff_positions_across_valid_CFs, n_features)
            using np.tile + advanced indexing.
        4.  Predict all perturbations in one batched call.
        5.  Group driver strings by CF neighbour.

        Returns
        -------
        sample_idx           : int
        per_neighbor_results : list of (cf_global_idx, sorted_driver_list)
        """
        orig      = orig_enc_sample
        orig_pred = self.model.predict([orig])[0]

        # Map BallTree-local indices to global row indices (pre-cached).
        global_ind = self.class_indices[int(opp_label)][ind_row]
        neighbors  = self.X_enc[global_ind]       # shape: (k, n_features)

        # Batch-verify all k CF candidates in one predict call.
        cf_preds     = self.model.predict(neighbors).ravel()
        valid_mask   = cf_preds == opp_label
        valid_global = global_ind[valid_mask]
        valid_neigh  = neighbors[valid_mask]       # shape: (n_valid, n_features)

        if len(valid_neigh) == 0:
            return sample_idx, []

        # Build perturbation matrix via NumPy broadcasting.
        # diff_matrix[i, j] is True where valid CF i differs from orig at j.
        # np.where returns aligned (cf_row, feat_col) index arrays.
        diff_matrix      = valid_neigh != orig          # (n_valid, n_feat)
        cf_row, feat_col = np.where(diff_matrix)        # both: shape (P,)
        n_perturbations  = len(cf_row)

        if n_perturbations == 0:
            return sample_idx, []

        # Build (P, n_features) matrix: start from P copies of orig, then
        # inject each CF value at the corresponding feature position.
        perturb_matrix = np.tile(orig, (n_perturbations, 1))
        perturb_matrix[np.arange(n_perturbations), feat_col] = \
            valid_neigh[cf_row, feat_col]

        # Predict all perturbations in one batched call.
        all_preds = self.model.predict(perturb_matrix).ravel()

        # Build driver strings for all perturbations (vectorised decode).
        driver_strings = [
            f'{self.feature_names[feat_col[p]]}'
            f'={self.feature_encoder.categories_[feat_col[p]][valid_neigh[cf_row[p], feat_col[p]]]}'
            for p in range(n_perturbations)
        ]

        # Collect drivers per valid CF neighbour.
        cf_drivers: dict[int, list] = {}
        for p, (pred, driver_str) in enumerate(zip(all_preds, driver_strings)):
            if pred != orig_pred:
                gidx = int(valid_global[cf_row[p]])
                cf_drivers.setdefault(gidx, []).append(driver_str)

        per_neighbor_results = [
            (gidx, sorted(drivers))
            for gidx, drivers in cf_drivers.items()
        ]

        return sample_idx, per_neighbor_results

    # ------------------------------------------------------------------
    # explain
    # ------------------------------------------------------------------

    def explain(self, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        """
        Run boundary-crossing analysis on (X, y).

        For each class label:
          1. Compute Hamming distance to the nearest opposite-class instance.
          2. Keep only instances whose distance ≤ the perc_threshold-th
             percentile (boundary filter).
          3. Keep only boundary instances the model predicts correctly,
             mirroring the consistency check in the original paper.
          4. For each surviving boundary instance, query k nearest CF
             neighbours and run the vectorised per-feature swap in parallel.

        Returns
        -------
        pd.DataFrame with columns:
            Sample_ID             — original DataFrame index
            CF_Neighbor_ID        — positional index in self.X_enc
            Counterfactual_Values — sorted list of 'FEATURE=cf_value' strings
        """
        print('  > Extracting counterfactuals '
              '(parallel, vectorised per-feature swap)...')

        X_enc = np.ascontiguousarray(
            self.feature_encoder.transform(X), dtype=np.int32
        )
        y_enc = self.label_encoder.transform(y)

        original_indices = X.index.tolist()
        rows = []

        all_classes = np.unique(y_enc)

        for label in all_classes:
            pos_idx = np.where(y_enc == label)[0]
            if len(pos_idx) == 0:
                continue

            opp_label = int(all_classes[all_classes != label][0])
            tree = self.trees.get(opp_label)
            if tree is None:
                continue

            # Boundary filter: keep instances within the perc_threshold-th
            # percentile of Hamming distance to the opposite class.
            min_dist, _ = tree.query(X_enc[pos_idx], k=1)
            min_dist    = min_dist.ravel()
            threshold   = np.percentile(min_dist, self.perc_threshold)

            boundary_idx = pos_idx[min_dist <= threshold]
            if len(boundary_idx) == 0:
                continue

            # Model-prediction filter: retain only instances predicted as
            # their true label (mirrors the original paper's consistency check).
            model_preds  = self.model.predict(X_enc[boundary_idx]).ravel()
            correct_mask = model_preds == label
            boundary_idx = boundary_idx[correct_mask]

            if len(boundary_idx) == 0:
                print(
                    f'    - class {label}: 0 boundary samples pass the '
                    f'model-prediction filter — skipping.'
                )
                continue

            print(
                f'    - class {label}: {len(boundary_idx)}/{len(pos_idx)} '
                f'boundary samples pass model-prediction filter '
                f'(perc_threshold={self.perc_threshold})'
            )

            # Query k nearest CF neighbours for each boundary instance.
            _, ind = tree.query(X_enc[boundary_idx], k=self.k_neighbors)

            # -------------------------------------------------------
            # Mega-batch approach: 2 model.predict() calls per class
            # instead of 3 × len(boundary_idx) individual calls.
            # Eliminates joblib serialisation overhead entirely.
            # -------------------------------------------------------
            B = len(boundary_idx)
            boundary_X = X_enc[boundary_idx]                    # (B, F)

            # (a) Batch-verify all B×k CF candidates in one predict.
            cf_global_all = self.class_indices[opp_label][ind]  # (B, k)
            cf_X_all      = self.X_enc[cf_global_all.ravel()]   # (B*k, F)
            cf_preds      = (self.model.predict(cf_X_all)
                             .ravel().reshape(B, -1))           # (B, k)
            valid_mask    = cf_preds == opp_label               # (B, k)

            n_valid = int(valid_mask.sum())
            if n_valid == 0:
                print(f'    - class {label}: 0 valid CF neighbours — skipping.')
                continue

            # (b) Vectorised perturbation matrix construction.
            valid_b, valid_cf_pos = np.where(valid_mask)
            cf_gidx_valid = cf_global_all[valid_b, valid_cf_pos]
            orig_valid    = boundary_X[valid_b]
            cf_valid      = self.X_enc[cf_gidx_valid]

            diff_matrix        = cf_valid != orig_valid
            pair_idx, feat_idx = np.where(diff_matrix)
            P = len(pair_idx)

            if P == 0:
                continue

            perturb_matrix = orig_valid[pair_idx].copy()
            perturb_matrix[np.arange(P), feat_idx] = \
                cf_valid[pair_idx, feat_idx]

            print(f'    - class {label}: {n_valid:,} valid CFs, '
                  f'{P:,} perturbations — predicting...')

            # (c) Mega-batch predict all perturbations.
            all_preds = self.model.predict(perturb_matrix).ravel()

            # (d) Identify drivers (prediction flipped from `label`).
            is_driver      = all_preds != label
            driver_indices = np.where(is_driver)[0]

            if len(driver_indices) == 0:
                continue

            driver_b       = valid_b[pair_idx[driver_indices]]
            driver_cf_gidx = cf_gidx_valid[pair_idx[driver_indices]]
            driver_feat    = feat_idx[driver_indices]
            driver_cf_val  = cf_valid[
                pair_idx[driver_indices], feat_idx[driver_indices]
            ].astype(int)

            driver_strs = [
                f'{self.feature_names[f]}'
                f'={self.feature_encoder.categories_[f][v]}'
                for f, v in zip(driver_feat, driver_cf_val)
            ]

            # Group drivers by (boundary sample, CF neighbour).
            groups: dict[tuple, list] = {}
            for d, ds in enumerate(driver_strs):
                key = (int(driver_b[d]), int(driver_cf_gidx[d]))
                groups.setdefault(key, []).append(ds)

            for (b_idx, cf_gidx), driver_list in groups.items():
                sample_idx = boundary_idx[b_idx]
                rows.append({
                    'Sample_ID':             original_indices[sample_idx],
                    'CF_Neighbor_ID':        cf_gidx,
                    'Counterfactual_Values': sorted(driver_list),
                })

        print(f'    - done: {len(rows)} (sample, CF-neighbour) pairs '
              f'with at least one driver\n')
        return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Post-processing helpers
# ---------------------------------------------------------------------------

def extract_labels_and_values(results_dir: Path) -> None:
    """
    Parse transactions_values.csv and write four derived CSV files:

        labels_only.csv          per (sample, CF): all driver feature names
        labels_only_unique.csv   per (sample, CF): unique driver feature names
        values_only.csv          per (sample, CF): all CF category values
        values_only_unique.csv   per (sample, CF): unique CF category values

    Deduplication is performed on (label, value) pairs jointly so that
    labels_only_unique and values_only_unique remain positionally aligned.
    """
    print('  > Extracting labels and values from counterfactual drivers...')
    input_file = results_dir / 'transactions_values.csv'

    if not input_file.exists():
        print('    - WARNING: transactions file not found, skipping.')
        return
    try:
        df = pd.read_csv(input_file)
    except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:
        print(f'    - WARNING: could not read transactions file ({exc}), skipping.')
        return
    if df.empty:
        print('    - WARNING: no transactions found, skipping.')
        return

    base_cols = ['Sample_ID']
    if 'CF_Neighbor_ID' in df.columns:
        base_cols.append('CF_Neighbor_ID')

    labels_list        = []
    labels_unique_list = []
    values_list        = []
    values_unique_list = []

    for _, row in df.iterrows():
        try:
            items = ast.literal_eval(str(row['Counterfactual_Values']))
        except (ValueError, SyntaxError):
            items = []

        labels, values = [], []
        for item in items:
            item_str = str(item)
            if '=' in item_str:
                lbl, val = item_str.split('=', 1)
                labels.append(lbl.strip())
                values.append(val.strip())

        # Deduplicate on (label, value) pairs jointly to keep the two lists
        # positionally aligned after deduplication.
        seen_pairs  = set()
        uniq_labels = []
        uniq_values = []
        for lbl, val in zip(labels, values):
            pair = (lbl, val)
            if pair not in seen_pairs:
                seen_pairs.add(pair)
                uniq_labels.append(lbl)
                uniq_values.append(val)

        labels_list.append(labels)
        labels_unique_list.append(uniq_labels)
        values_list.append(values)
        values_unique_list.append(uniq_values)

    base_data = {col: df[col] for col in base_cols}

    for filename, col_name, data in [
        ('labels_only.csv',        'Labels', labels_list),
        ('labels_only_unique.csv', 'Labels', labels_unique_list),
        ('values_only.csv',        'Values', values_list),
        ('values_only_unique.csv', 'Values', values_unique_list),
    ]:
        pd.DataFrame({**base_data, col_name: data}).to_csv(
            results_dir / filename, index=False
        )
        print(f'    - saved {filename}')


def aggregate_drivers_by_sample(results_dir: Path) -> None:
    """
    Collapse all (sample, CF-neighbour) rows into one transaction per sample,
    writing two aggregated CSV files for ARM on feature labels.

    aggregated_labels_by_sample.csv
        One row per sample.  'Labels' contains the union of all unique driver
        feature names across every CF neighbour — duplicates removed.
        Recommended input for FP-Growth.

    aggregated_labels_duplicates_by_sample.csv
        One row per sample.  'Labels' contains all driver feature names
        including duplicates (a feature appearing as a driver for multiple CFs
        of the same sample is listed multiple times).

    Both files share columns:
        Sample_ID        — original row index in the dataset
        Labels           — list of feature-name drivers (see above)
        Num_Labels       — cardinality of the label list
        Num_CF_Neighbors — number of CF neighbours that contributed drivers
    """
    print('  > Aggregating labels by sample...')

    labels_path = results_dir / 'labels_only.csv'
    out_unique  = results_dir / 'aggregated_labels_by_sample.csv'
    out_dupl    = results_dir / 'aggregated_labels_duplicates_by_sample.csv'

    if not labels_path.exists():
        print(f'    - WARNING: {labels_path.name} not found, skipping.')
        return
    try:
        df = pd.read_csv(labels_path)
    except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:
        print(f'    - WARNING: could not read labels file ({exc}), skipping.')
        return
    if df.empty:
        print('    - WARNING: labels file is empty, skipping.')
        return

    df['Labels'] = df['Labels'].apply(ast.literal_eval)

    aggregated_unique = []
    aggregated_dupl   = []

    for sample_id, group in df.groupby('Sample_ID'):
        n_cf = len(group)

        # Union of all driver labels across CFs — duplicates removed.
        all_labels_set = set()
        for labels_list in group['Labels']:
            all_labels_set.update(labels_list)
        unique_labels = sorted(all_labels_set)

        # All driver labels concatenated across CFs — duplicates kept.
        all_labels_list = sorted(
            lbl for labels_list in group['Labels'] for lbl in labels_list
        )

        aggregated_unique.append({
            'Sample_ID':        sample_id,
            'Labels':           unique_labels,
            'Num_Labels':       len(unique_labels),
            'Num_CF_Neighbors': n_cf,
        })
        aggregated_dupl.append({
            'Sample_ID':        sample_id,
            'Labels':           all_labels_list,
            'Num_Labels':       len(all_labels_list),
            'Num_CF_Neighbors': n_cf,
        })

    unique_df = pd.DataFrame(aggregated_unique)
    dupl_df   = pd.DataFrame(aggregated_dupl)

    unique_df.to_csv(out_unique, index=False)
    dupl_df.to_csv(out_dupl,    index=False)

    n_samples = len(unique_df)
    print(f'    - {len(df)} (sample, CF) pairs collapsed into '
          f'{n_samples} unique samples')

    for tag, frame in [('unique labels', unique_df), ('labels with duplicates', dupl_df)]:
        print(f'    [{tag} per sample]')
        for n_labels, count in frame['Num_Labels'].value_counts().sort_index().items():
            pct = count / n_samples * 100
            print(f'      {n_labels} label(s): {count} samples ({pct:.1f}%)')

    print(f'    - saved {out_unique.name}')
    print(f'    - saved {out_dupl.name}')


# ---------------------------------------------------------------------------
# Experiment runner
# ---------------------------------------------------------------------------

def run_for_k_values(
    k_values: list[int],
    data_path: Path,
    output_base_dir: Path,
    target_col: str  = 'INCOME_ABOVE_THRESHOLD',
    perc_threshold: int = 10,
) -> dict[int, Path]:
    """
    Train CategoricalBoCSoR once and run counterfactual extraction for each k.

    The model and BallTrees are fitted on the training split and shared across
    all k values; only the neighbourhood query size changes per iteration.

    Parameters
    ----------
    k_values        : neighbourhood sizes to evaluate
    data_path       : path to the input CSV
    output_base_dir : root directory for per-k result sub-folders
    target_col      : name of the target column in the CSV
    perc_threshold  : boundary filter percentile

    Returns
    -------
    dict mapping k → Path of labels_only_unique.csv for that k
    """
    output_base_dir = Path(output_base_dir)
    output_base_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n{"=" * 70}')
    print(f'K-VARIATION EXPERIMENT — {len(k_values)} k values')
    print(f'{"=" * 70}')
    print(f'  > k values        : {k_values}')
    print(f'  > perc_threshold  : {perc_threshold}')
    print(f'  > target column   : {target_col}')
    print('-' * 50)

    print('  > Loading dataset and splitting...')
    df = pd.read_csv(data_path)

    if target_col != 'target':
        df = df.rename(columns={target_col: 'target'})

    X, y = df.drop(columns=['target']), df['target']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f'    - train: {len(X_tr):,} samples  |  test: {len(X_te):,} samples')

    # Fit once — model and BallTrees are shared across all k iterations.
    print('\n  > Fitting model (shared across all k values)...')
    explainer = CategoricalBoCSoR(
        k_neighbors=k_values[0], perc_threshold=perc_threshold
    )
    explainer.fit(X_tr, y_tr)

    k_labels_map: dict[int, Path] = {}

    for i, k in enumerate(k_values):
        k_dir = output_base_dir / f'k_{k}'
        k_dir.mkdir(parents=True, exist_ok=True)

        print(f'\n  [{i + 1}/{len(k_values)}] k = {k}')
        explainer.k_neighbors = k

        # explain() runs on the training set: we explain the classifier's
        # decision logic, not its generalisation to unseen instances.
        transactions = explainer.explain(X_tr, y_tr)

        if transactions.empty:
            print('    > 0 transactions — skipping downstream steps.')
            continue

        transactions_path = k_dir / 'transactions_values.csv'
        transactions.to_csv(transactions_path, index=False)
        print(f'    > {len(transactions)} transactions saved to '
              f'{transactions_path.name}')

        extract_labels_and_values(k_dir)
        aggregate_drivers_by_sample(k_dir)

        labels_path = k_dir / 'labels_only_unique.csv'
        if labels_path.exists() and labels_path.stat().st_size > 0:
            k_labels_map[k] = labels_path
        else:
            print(f'    - WARNING: labels_only_unique.csv is empty for k={k}, '
                  f'skipping.')

    print(f'\n{"=" * 70}')
    print('  > All k values completed.')
    print(f'{"=" * 70}\n')

    return k_labels_map


# ---------------------------------------------------------------------------
# Log capturing utility
# ---------------------------------------------------------------------------

class _TeeWriter:
    """Write to the original stdout and simultaneously capture in a StringIO buffer."""

    def __init__(self, original_stdout):
        self._orig = original_stdout
        self._buf  = io.StringIO()

    def write(self, text: str) -> None:
        self._orig.write(text)
        self._buf.write(text)

    def flush(self) -> None:
        self._orig.flush()

    def getvalue(self) -> str:
        return self._buf.getvalue()


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def main(
    survey_year: str       = '2024',
    regions: dict          = None,
    k_values: list[int]    = None,
    perc_threshold: int    = 10,
    target_col: str        = 'INCOME_ABOVE_THRESHOLD',
    base_dir: Path         = None,
) -> None:
    """
    Run counterfactual extraction for all specified regions and k values.

    Parameters
    ----------
    survey_year     : ACS survey year, used to locate input CSV files.
    regions         : mapping of region name → CSV path.  If None, defaults
                      to Northeast and South under base_dir/data/.
    k_values        : neighbourhood sizes for BoCSoR.
    perc_threshold  : boundary filter percentile.
    target_col      : name of the binary target column in the CSV.
    base_dir        : project root directory.  Defaults to two levels above
                      this file when run standalone.
    """
    if base_dir is None:
        if Path('/kaggle/working').exists():
            base_dir = Path('/kaggle/working')
        elif Path('/content').exists():
            base_dir = Path('/content')
        else:
            base_dir = Path(__file__).resolve().parent.parent
    base_dir = Path(base_dir)

    if k_values is None:
        k_values = [1, 3, 5, 7]

    if regions is None:
        data_dir = base_dir / 'data'
        regions  = {
            'northeast': data_dir / f'acs_income_northeast_{survey_year}.csv',
            'south':     data_dir / f'acs_income_south_{survey_year}.csv',
        }

    results_dir = base_dir / 'results'

    tee = _TeeWriter(sys.stdout)
    sys.stdout = tee

    try:
        for region, data_path in regions.items():
            output_dir = results_dir / region / 'important_features'
            output_dir.mkdir(parents=True, exist_ok=True)

            print('\n' + '=' * 70)
            print(f'COUNTERFACTUAL EXTRACTION — {region.upper()}')
            print('=' * 70 + '\n')

            if not Path(data_path).exists():
                print(f'  > Error: {data_path} not found — '
                      f'run create_dataset.py first.')
                continue

            k_labels_map = run_for_k_values(
                k_values       = k_values,
                data_path      = data_path,
                output_base_dir= output_dir,
                target_col     = target_col,
                perc_threshold = perc_threshold,
            )

            print('  > k_labels_map ready:')
            for k, path in k_labels_map.items():
                print(f'    k={k:>2} -> {path}')

        print('\n' + '=' * 70)
        print('Done.')
        print('=' * 70 + '\n')

    finally:
        sys.stdout = tee._orig

    # Save full execution log and per-region excerpts.
    full_log = tee.getvalue()
    results_dir.mkdir(parents=True, exist_ok=True)

    global_log = results_dir / 'feature_importance_log.txt'
    global_log.write_text(full_log, encoding='utf-8')
    print(f'  > Full log saved to {global_log}')

    for region in regions:
        region_dir = results_dir / region / 'important_features'
        if region_dir.exists():
            marker = f'COUNTERFACTUAL EXTRACTION — {region.upper()}'
            start  = full_log.find(marker)
            if start != -1:
                next_start = full_log.find(
                    'COUNTERFACTUAL EXTRACTION', start + len(marker)
                )
                snippet = (
                    full_log[start:next_start]
                    if next_start != -1
                    else full_log[start:]
                )
                (region_dir / 'feature_importance_log.txt').write_text(
                    snippet, encoding='utf-8'
                )
                print(f'  > Region log saved to '
                      f'{region_dir / "feature_importance_log.txt"}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile src/macroscopic_experiment_association_rules.py
"""
macroscopic_experiment_association_rules.py
===========================================
Runs FP-Growth association-rule mining on the feature-driver transactions
produced by feature_importance.py, searching a grid of support, confidence
and lift thresholds for each value of k.

The neutral lift window [1 - half_window, 1 + half_window] is excluded
everywhere (FP-Growth filtering and heatmap masking) via a single helper
(_neutral_window) to guarantee consistency between the two stages.
Negative correlations (lift < neutral_lo) are preserved and analysed.

Public API
----------
run_k_comparison(k_labels_map, output_dir, auto_calibrate, ...)
    Run explore_association_rules for each k and build a cross-k comparison.

explore_association_rules(df, output_dir, ...)
    Full support × confidence × lift grid search; writes rules and heatmaps.

calibrate_parameters(encoded_df, ...)
    Auto-calibrate grid bounds from item-support frequencies.
"""

import ast
import datetime
import os
import platform
import shutil
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from mlxtend.frequent_patterns import association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

# Headless backend: required on servers without a display.
matplotlib.use('Agg')

# ---------------------------------------------------------------------------
# Hardware-aware parallelism
# ---------------------------------------------------------------------------

_CPU_CORES = os.cpu_count() or 1

# On Apple Silicon, cap n_jobs at the P-core count to avoid the E-core
# straggler bottleneck.  Adjust _PERF_CORES manually if needed
# (M2 base=4, Pro=6/8, Max=8–12, Ultra=16).
# On Linux/Windows all cores are equivalent.
_PERF_CORES = 4 if platform.system() == 'Darwin' else _CPU_CORES


# ---------------------------------------------------------------------------
# Neutral-window helper — single source of truth
# ---------------------------------------------------------------------------

def _neutral_window(lift_neutral_half_window: float) -> tuple[float, float]:
    """
    Return (lo, hi) for the neutral lift window.

    Rules with lift in [lo, hi] are excluded in both FP-Growth filtering and
    heatmap masking.  Using this single helper guarantees the two stages agree
    regardless of the lift_delta step size.

    Formula
    -------
    lo = round(1.0 - lift_neutral_half_window, 4)
    hi = round(1.0 + lift_neutral_half_window, 4)

    With the default half_window=0.25 this gives [0.75, 1.25].

    Negative correlations
    ---------------------
    Rules with lift < lo indicate features that tend NOT to co-occur on the
    decision boundary — an analytically meaningful anti-correlation.  Only
    the [lo, hi] band is removed; rules below lo are intentionally preserved.
    Set lift_min=0.0 in the entry point to include them in the grid search.
    """
    lo = round(1.0 - lift_neutral_half_window, 4)
    hi = round(1.0 + lift_neutral_half_window, 4)
    return lo, hi


# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def extract_labels(labels_only_path: Path) -> pd.DataFrame:
    """
    Read the labels CSV and one-hot-encode each transaction for FP-Growth.

    Supported file formats:
        aggregated_labels_by_sample.csv  — one row per sample
        labels_only_unique.csv           — one row per (sample, CF) pair

    Both share a 'Labels' column containing Python list literals
    (e.g. "['OCCP', 'SCHL']") parsed with ast.literal_eval.

    Returns a Boolean-encoded DataFrame (rows = transactions, columns = items).
    """
    print('  > Loading and encoding labels...')
    df = pd.read_csv(labels_only_path)

    if 'Labels' not in df.columns:
        raise ValueError(
            f"CSV must have a 'Labels' column. "
            f"Found: {df.columns.tolist()}  |  file: {labels_only_path}"
        )

    print(f'    (file: {Path(labels_only_path).name}, {len(df):,} rows)')
    itemsets = df['Labels'].apply(ast.literal_eval)

    te     = TransactionEncoder()
    te_ary = te.fit(itemsets).transform(itemsets)
    enc_df = pd.DataFrame(te_ary, columns=te.columns_)

    print('  > First few encoded rows:')
    print(enc_df.head())
    print('-' * 50)

    return enc_df


# ---------------------------------------------------------------------------
# Filesystem helpers
# ---------------------------------------------------------------------------

def cleanup_empty_folders(output_dir: Path) -> tuple[int, int]:
    """
    Remove conf_* folders that contain no rules and sup_* folders that have
    no remaining conf_* children.

    Returns (n_conf_removed, n_sup_removed).
    """
    output_dir   = Path(output_dir)
    removed_conf = 0
    removed_sup  = 0

    for sup_dir in sorted(output_dir.glob('sup_*')):
        if not sup_dir.is_dir():
            continue
        for conf_dir in sorted(sup_dir.glob('conf_*')):
            if not conf_dir.is_dir():
                continue
            rules_csv    = conf_dir / 'rules.csv'
            should_remove = not rules_csv.exists()
            if not should_remove:
                try:
                    should_remove = pd.read_csv(rules_csv).empty
                except Exception:
                    should_remove = True
            if should_remove:
                shutil.rmtree(conf_dir)
                removed_conf += 1
        if not list(sup_dir.glob('conf_*')):
            shutil.rmtree(sup_dir)
            removed_sup += 1

    return removed_conf, removed_sup


# ---------------------------------------------------------------------------
# In-memory grid search
# ---------------------------------------------------------------------------

def grid_search_fpgrowth_delta(
    df,
    sup_min, sup_max, sup_delta,
    conf_min, conf_max, conf_delta,
    lift_min, lift_max, lift_delta,
    lift_neutral_half_window: float = 0.25,
) -> pd.DataFrame:
    """
    Quick in-memory grid search over support × confidence × lift.

    No files are written; use explore_association_rules() for full output.
    Returns a summary DataFrame sorted by (Number_of_Rules DESC, Lift DESC).
    """
    print(f'\n{"=" * 70}')
    print('GRID SEARCH: FP-GROWTH (IN-MEMORY)')
    print(f'{"=" * 70}')

    support_grid    = np.round(np.arange(sup_min,  sup_max  + sup_delta  / 2, sup_delta),  4)
    confidence_grid = np.round(np.arange(conf_min, conf_max + conf_delta / 2, conf_delta), 4)
    lift_grid       = np.round(np.arange(lift_min, lift_max + lift_delta / 2, lift_delta), 4)

    lift_neutral_lo, lift_neutral_hi = _neutral_window(lift_neutral_half_window)

    print(f'  > support grid    : {support_grid}')
    print(f'  > confidence grid : {confidence_grid}')
    print(f'  > lift grid       : {lift_grid}')
    print(f'  > neutral window (excluded): [{lift_neutral_lo}, {lift_neutral_hi}]')
    print('-' * 50)

    results = []

    for min_sup in support_grid:
        frequent_itemsets = fpgrowth(df, min_support=min_sup, use_colnames=True)
        if len(frequent_itemsets) == 0:
            continue

        # Single association_rules() call at the lowest confidence threshold,
        # then filter in-memory — avoids redundant FP-Growth runs per conf.
        try:
            all_rules = association_rules(
                frequent_itemsets,
                metric='confidence',
                min_threshold=float(confidence_grid[0]),
            )
        except ValueError:
            continue

        if len(all_rules) == 0:
            continue

        # Exclude neutral window; preserve negative correlations (lift < lo).
        all_rules = all_rules[
            (all_rules['lift'] < lift_neutral_lo) |
            (all_rules['lift'] > lift_neutral_hi)
        ]
        if len(all_rules) == 0:
            continue

        for min_conf in confidence_grid:
            rules = all_rules[all_rules['confidence'] >= min_conf]
            if len(rules) == 0:
                continue

            for min_lift in lift_grid:
                if lift_neutral_lo <= min_lift <= lift_neutral_hi:
                    continue
                filtered = rules[rules['lift'] >= min_lift]
                n = len(filtered)
                results.append({
                    'Support':         min_sup,
                    'Confidence':      min_conf,
                    'Lift':            min_lift,
                    'Number_of_Rules': n,
                    'Max_Lift':        round(filtered['lift'].max(), 4) if n > 0 else 0,
                    'Mean_Confidence': round(filtered['confidence'].mean(), 4) if n > 0 else 0,
                })

    print('  > Done.')

    result_df = pd.DataFrame(results)
    if not result_df.empty:
        result_df = result_df.sort_values(
            by=['Number_of_Rules', 'Lift'], ascending=[False, False]
        )
    return result_df


# ---------------------------------------------------------------------------
# Heatmaps
# ---------------------------------------------------------------------------

def plot_heatmaps(
    summary_df: pd.DataFrame,
    output_dir: Path,
    lift_display_step: float        = 0.1,
    lift_neutral_half_window: float = 0.25,
    lift_delta: float               = 0.05,
) -> None:
    """
    Generate three heatmaps — support-confidence, support-lift, and
    confidence-lift — each cell showing the maximum rule count over the
    third parameter (darker = more rules).

    Lift-axis masking uses _neutral_window() (the same helper used during
    FP-Growth filtering), guaranteeing exact alignment between the two stages.
    Negative-correlation columns (lift < neutral_lo) are preserved and appear
    on the left side of the lift axis when lift_min=0.0.

    Parameters
    ----------
    lift_delta  : kept for API compatibility; window computation delegates
                  to _neutral_window(lift_neutral_half_window).
    """
    if summary_df.empty:
        print('  > Summary is empty, skipping heatmaps.')
        return

    output_dir  = Path(output_dir)
    heatmap_dir = output_dir / 'heatmaps'
    heatmap_dir.mkdir(parents=True, exist_ok=True)

    print('  > Generating heatmaps...')

    df = summary_df.copy()
    df['Lift_display'] = (
        (df['Lift_threshold'] / lift_display_step).round() * lift_display_step
    ).round(4)

    neutral_lo, neutral_hi = _neutral_window(lift_neutral_half_window)

    configs = [
        ('Confidence',   'Support',    'support_confidence', False),
        ('Lift_display', 'Support',    'support_lift',        True),
        ('Lift_display', 'Confidence', 'confidence_lift',     True),
    ]

    for x_col, y_col, suffix, x_is_lift in configs:
        pivot = (
            df.groupby([y_col, x_col])['Number_of_Rules']
            .max()
            .unstack(level=x_col)
            .sort_index(ascending=False)
            .fillna(0)
            .astype(int)
        )

        if x_is_lift:
            # Mask neutral window; negative-correlation columns survive.
            pivot = pivot.loc[
                :, ~pivot.columns.to_series().between(
                    neutral_lo, neutral_hi, inclusive='both'
                )
            ]
            if (pivot != 0).any(axis=0).any():
                last_nz = int(np.where((pivot != 0).any(axis=0).values)[0].max())
                pivot   = pivot.iloc[:, :last_nz + 1]

        n_cols = len(pivot.columns)
        n_rows = len(pivot.index)
        fig, ax = plt.subplots(
            figsize=(max(10, n_cols * 0.75), max(4, n_rows * 0.55))
        )

        img = ax.imshow(
            pivot.values, aspect='auto', cmap='YlOrBr', interpolation='nearest'
        )

        ax.set_xticks(range(n_cols))
        ax.set_xticklabels(
            [f'{v:.2f}' for v in pivot.columns],
            rotation=40, ha='right', fontsize=8,
        )
        ax.set_yticks(range(n_rows))
        ax.set_yticklabels([f'{v:.2f}' for v in pivot.index], fontsize=8)

        x_label = 'Lift' if x_is_lift else x_col
        ax.set_xlabel(x_label, fontsize=11, labelpad=8)
        ax.set_ylabel(y_col,   fontsize=11, labelpad=8)
        ax.set_title(
            f'Max Number of Rules — {y_col} vs {x_label}\n'
            f'(darker = more rules; max over the third parameter)',
            fontsize=11, pad=14,
        )

        max_val = pivot.values.max() if pivot.values.max() > 0 else 1
        for ri in range(n_rows):
            for ci in range(n_cols):
                val = pivot.values[ri, ci]
                if val > 0:
                    txt_color = 'white' if (val / max_val) > 0.55 else 'black'
                    ax.text(
                        ci, ri, str(val),
                        ha='center', va='center', fontsize=7, color=txt_color,
                    )

        cbar = plt.colorbar(img, ax=ax, fraction=0.025, pad=0.02)
        cbar.set_label('Number of Rules', fontsize=9)
        plt.tight_layout()

        fig.savefig(
            heatmap_dir / f'heatmap_{suffix}.png', dpi=150, bbox_inches='tight'
        )
        plt.close(fig)
        print(f'    > saved heatmaps/heatmap_{suffix}.png')

    print(f'  > heatmaps saved to {heatmap_dir}/')


# ---------------------------------------------------------------------------
# Core worker — parallelised over support thresholds
# ---------------------------------------------------------------------------

def _process_one_support(
    min_sup: float,
    sup_idx: int,
    n_sup: int,
    df,
    output_dir: Path,
    confidence_grid,
    lift_grid_used,
    lift_window_lo: float,
    lift_window_hi: float,
) -> list:
    """
    Process one support threshold for explore_association_rules.

    Called in parallel; writes to its own sup_{min_sup}/ subdirectory.
    Returns a list of summary-row dicts for the calling process to aggregate.

    Both A→B and B→A rule directions are retained (they have different
    confidence values and carry independent information).
    conviction=inf (confidence=1.0) is replaced with np.nan before saving.

    Column semantics
    ----------------
    support_raw       proportion as decimal string (trailing zeros preserved)
    support_pct       proportion as percentage (float, 2 d.p.)
    confidence_raw    P(consequent | antecedent) as decimal string
    confidence_pct    P(consequent | antecedent) as percentage
    lift              ratio (1.0=independence, >1 positive, <1 negative)
    leverage          support(A∪B) − support(A)·support(B)
    conviction        directional strength (inf → NaN when confidence=1.0)
    """
    output_dir = Path(output_dir)
    sup_label  = f'{min_sup:.2f}'
    sup_dir    = output_dir / f'sup_{sup_label}'
    sup_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n  [{sup_idx}/{n_sup}] support = {min_sup}')
    print('    > running FP-Growth...')

    frequent_itemsets = fpgrowth(df, min_support=min_sup, use_colnames=True)
    print(f'    > {len(frequent_itemsets)} frequent itemsets found')

    if len(frequent_itemsets) == 0:
        print('    > no frequent itemsets, skipping.')
        return []

    fi = frequent_itemsets.copy()
    fi['itemset_str']    = fi['itemsets'].apply(lambda x: ', '.join(sorted(x)))
    fi['itemset_length'] = fi['itemsets'].apply(len)
    fi = fi[['itemset_str', 'itemset_length', 'support']]
    fi = fi.sort_values(by=['itemset_length', 'support'], ascending=[True, False])
    fi.to_csv(sup_dir / 'frequent_itemsets.csv', index=False)

    itemsets_by_len = fi['itemset_length'].value_counts().sort_index().to_dict()
    print(f'    > breakdown: '
          f'{", ".join(f"len={k}: {v}" for k, v in itemsets_by_len.items())}')

    with open(sup_dir / 'frequent_itemsets_summary.txt', 'w') as f:
        f.write('Frequent Itemsets Summary\n')
        f.write(f'{"=" * 60}\n\n')
        f.write(f'Parameters:\n  Min Support: {min_sup}\n\n')
        f.write(f'Results:\n  Total: {len(frequent_itemsets)}\n')
        for length, count in itemsets_by_len.items():
            f.write(f'  len={length}: {count}\n')
        f.write(f'\nAll Frequent Itemsets (by length, then support desc):\n{"-" * 60}\n')
        for _, row in fi.iterrows():
            f.write(
                f'  [{row["itemset_str"]}]  '
                f'support={row["support"]:.4f}  '
                f'length={row["itemset_length"]}\n'
            )

    local_summary_rows = []

    # Single association_rules() call at the lowest confidence threshold;
    # subsequent confidence levels are obtained by in-memory filtering.
    try:
        all_rules = association_rules(
            frequent_itemsets,
            metric='confidence',
            min_threshold=float(confidence_grid[0]),
        )
    except ValueError:
        return local_summary_rows

    if len(all_rules) == 0:
        return local_summary_rows

    # Exclude neutral window; preserve negative correlations (lift < lo).
    all_rules = all_rules[
        (all_rules['lift'] < lift_window_lo) |
        (all_rules['lift'] > lift_window_hi)
    ]
    all_rules = all_rules.sort_values('lift', ascending=False).reset_index(drop=True)

    if len(all_rules) == 0:
        return local_summary_rows

    for min_conf in confidence_grid:
        conf_label = f'{min_conf:.2f}'
        conf_dir   = sup_dir / f'conf_{conf_label}'

        rules = all_rules[all_rules['confidence'] >= min_conf].reset_index(drop=True)
        if len(rules) == 0:
            continue

        # conviction=inf (confidence=1.0) → NaN; computed once and shared.
        conviction_vals = rules['conviction'].replace([np.inf, -np.inf], np.nan)

        # Compact output.
        fmt = pd.DataFrame()
        fmt['antecedents']    = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
        fmt['consequents']    = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
        fmt['support_raw']    = [f'{v:.4f}' for v in rules['support']]
        fmt['support_pct']    = (rules['support']    * 100).round(2)
        fmt['confidence_raw'] = [f'{v:.4f}' for v in rules['confidence']]
        fmt['confidence_pct'] = (rules['confidence'] * 100).round(2)
        fmt['lift']           = rules['lift'].round(4)
        fmt['leverage']       = rules['leverage'].round(6)
        fmt['conviction']     = conviction_vals.round(4)

        # Detailed output.
        det = pd.DataFrame()
        det['antecedents']            = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
        det['consequents']            = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
        det['antecedent_length']      = rules['antecedents'].apply(len)
        det['consequent_length']      = rules['consequents'].apply(len)
        det['rule_length']            = det['antecedent_length'] + det['consequent_length']
        det['antecedent_support_raw'] = [f'{v:.4f}' for v in rules['antecedent support']]
        det['antecedent_support_pct'] = (rules['antecedent support'] * 100).round(2)
        det['consequent_support_raw'] = [f'{v:.4f}' for v in rules['consequent support']]
        det['consequent_support_pct'] = (rules['consequent support'] * 100).round(2)
        det['support_raw']            = [f'{v:.4f}' for v in rules['support']]
        det['support_pct']            = (rules['support']    * 100).round(2)
        det['confidence_raw']         = [f'{v:.4f}' for v in rules['confidence']]
        det['confidence_pct']         = (rules['confidence'] * 100).round(2)
        det['lift']                   = rules['lift'].round(4)
        det['leverage']               = rules['leverage'].round(6)
        det['conviction']             = conviction_vals.round(4)

        conf_dir.mkdir(parents=True, exist_ok=True)
        fmt.to_csv(conf_dir / 'rules.csv',          index=False)
        det.to_csv(conf_dir / 'rules_detailed.csv', index=False)

        with open(conf_dir / 'summary.txt', 'w') as f:
            f.write('Association Rules Summary\n')
            f.write(f'{"=" * 60}\n\n')
            f.write('Parameters:\n')
            f.write(f'  Min Support:    {min_sup}\n')
            f.write(f'  Min Confidence: {min_conf}\n')
            f.write(f'  Neutral Lift Window (excluded): '
                    f'[{lift_window_lo}, {lift_window_hi}]\n\n')
            f.write('Results:\n')
            f.write(f'  Frequent Itemsets: {len(frequent_itemsets)}\n')
            f.write(f'  Association Rules: {len(rules)}\n\n')
            f.write('Statistics:\n')
            f.write(f'  Avg Support:     {rules["support"].mean() * 100:.2f}%\n')
            f.write(f'  Avg Confidence:  {rules["confidence"].mean() * 100:.2f}%\n')
            f.write(f'  Avg Lift:        {rules["lift"].mean():.4f}\n')
            f.write(f'  Lift Range:      {rules["lift"].min():.4f} — '
                    f'{rules["lift"].max():.4f}\n')
            f.write(f'  Avg Leverage:    {rules["leverage"].mean():.6f}\n')
            f.write(f'  Avg Rule Length: {det["rule_length"].mean():.2f}\n\n')
            n_inf = conviction_vals.isna().sum()
            if n_inf > 0:
                f.write(f'  Note: {n_inf} rule(s) have confidence=1.0 '
                        f'(conviction=inf → saved as NaN)\n\n')
            f.write(f'Top 10 Rules (by Lift):\n{"-" * 60}\n')
            for idx, row in fmt.head(10).iterrows():
                f.write(
                    f'{idx + 1}. {row["antecedents"]} => {row["consequents"]}\n'
                    f'   support={row["support_pct"]:.2f}% | '
                    f'confidence={row["confidence_pct"]:.2f}% | '
                    f'lift={row["lift"]:.4f} | '
                    f'leverage={row["leverage"]:.6f}\n\n'
                )

        print(f'    > [conf={min_conf}] {len(rules)} rules saved to '
              f'{conf_dir.relative_to(output_dir)}/')

        for min_lift in lift_grid_used:
            filtered = rules[rules['lift'] >= min_lift]
            n = len(filtered)
            rl = (
                filtered['antecedents'].apply(len) + filtered['consequents'].apply(len)
            ) if n > 0 else pd.Series(dtype=float)

            local_summary_rows.append({
                'Support':               min_sup,
                'Confidence':            min_conf,
                'Lift_threshold':        min_lift,
                'Number_of_Rules':       n,
                'Max_Lift':              round(filtered['lift'].max(), 4) if n > 0 else 0.0,
                'Min_Lift':              round(filtered['lift'].min(), 4) if n > 0 else 0.0,
                'Avg_Lift':              round(filtered['lift'].mean(), 4) if n > 0 else 0.0,
                'Avg_Confidence':        round(filtered['confidence'].mean(), 4) if n > 0 else 0.0,
                'Avg_Support':           round(filtered['support'].mean(), 4) if n > 0 else 0.0,
                'Avg_Rule_Length':       round(rl.mean(), 4) if n > 0 else 0.0,
                'Max_Rule_Length':       int(rl.max()) if n > 0 else 0,
                'Num_Frequent_Itemsets': len(frequent_itemsets),
                'Num_FI_length_1':       itemsets_by_len.get(1, 0),
                'Num_FI_length_2':       itemsets_by_len.get(2, 0),
                'Num_FI_length_3plus':   sum(v for k, v in itemsets_by_len.items() if k >= 3),
            })

    return local_summary_rows


# ---------------------------------------------------------------------------
# Full grid exploration
# ---------------------------------------------------------------------------

def explore_association_rules(
    df,
    output_dir: Path,
    sup_min, sup_max, sup_delta,
    conf_min, conf_max, conf_delta,
    lift_min, lift_max, lift_delta,
    lift_neutral_half_window: float = 0.25,
) -> pd.DataFrame:
    """
    Full grid search over support × confidence × lift using FP-Growth.

    Workers are dispatched in parallel over support thresholds (loky backend).
    The neutral lift window is excluded via _neutral_window().
    Negative correlations and both A→B / B→A rule directions are retained.

    Returns a summary DataFrame sorted by (Number_of_Rules DESC, Max_Lift DESC).
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    support_grid    = np.round(np.arange(sup_min,  sup_max  + sup_delta  / 2, sup_delta),  4)
    confidence_grid = np.round(np.arange(conf_min, conf_max + conf_delta / 2, conf_delta), 4)
    lift_grid       = np.round(np.arange(lift_min, lift_max + lift_delta / 2, lift_delta), 4)

    lift_window_lo, lift_window_hi = _neutral_window(lift_neutral_half_window)
    lift_grid_used  = [v for v in lift_grid
                       if not (lift_window_lo <= v <= lift_window_hi)]
    total_combos    = len(support_grid) * len(confidence_grid) * len(lift_grid_used)

    print(f'\n{"=" * 70}')
    print('FULL EXPLORATION: FP-GROWTH ASSOCIATION RULES')
    print(f'{"=" * 70}')
    print(f'  > support    : {len(support_grid)} values '
          f'[{support_grid[0]} ... {support_grid[-1]}, step={sup_delta}]')
    print(f'  > confidence : {len(confidence_grid)} values '
          f'[{confidence_grid[0]} ... {confidence_grid[-1]}, step={conf_delta}]')
    print(f'  > lift       : {len(lift_grid_used)} values used '
          f'(of {len(lift_grid)} total, '
          f'{len(lift_grid) - len(lift_grid_used)} skipped — '
          f'neutral window [{lift_window_lo}, {lift_window_hi}], step={lift_delta})')
    print(f'  > total combinations: {total_combos:,}')
    print('-' * 50)

    n_jobs = min(_PERF_CORES, len(support_grid))
    print(f'  > Launching parallel FP-Growth over {len(support_grid)} support '
          f'values (n_jobs={n_jobs} of {_CPU_CORES} logical / '
          f'{_PERF_CORES} perf cores)')

    parallel_results = Parallel(n_jobs=n_jobs, backend='loky', verbose=0)(
        delayed(_process_one_support)(
            min_sup        = min_sup,
            sup_idx        = sup_idx,
            n_sup          = len(support_grid),
            df             = df,
            output_dir     = output_dir,
            confidence_grid= confidence_grid,
            lift_grid_used = lift_grid_used,
            lift_window_lo = lift_window_lo,
            lift_window_hi = lift_window_hi,
        )
        for sup_idx, min_sup in enumerate(support_grid, start=1)
    )

    summary_rows = [row for worker_rows in parallel_results for row in worker_rows]

    print(f'\n{"=" * 70}')
    print('  > Exploration complete, building summary...')

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(
            by=['Number_of_Rules', 'Max_Lift'], ascending=[False, False]
        ).reset_index(drop=True)

    summary_df.to_csv(output_dir / 'summary.csv', index=False)
    combos_with_rules = (
        int((summary_df['Number_of_Rules'] > 0).sum())
        if not summary_df.empty else 0
    )

    with open(output_dir / 'exploration_summary.txt', 'w') as f:
        f.write('FULL EXPLORATION SUMMARY\n')
        f.write(f'{"=" * 70}\n\n')
        f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
        f.write('Parameter Grids:\n')
        f.write(f'  Support    : {len(support_grid)} values '
                f'[{support_grid[0]} ... {support_grid[-1]}, step={sup_delta}]\n')
        f.write(f'  Confidence : {len(confidence_grid)} values '
                f'[{confidence_grid[0]} ... {confidence_grid[-1]}, step={conf_delta}]\n')
        f.write(f'  Lift       : {len(lift_grid_used)} values used '
                f'(of {len(lift_grid)} total, '
                f'neutral window [{lift_window_lo}, {lift_window_hi}] excluded), '
                f'step={lift_delta}\n\n')
        f.write('Results:\n')
        f.write(f'  Total combinations : {total_combos:,}\n')
        f.write(f'  With >= 1 rule     : {combos_with_rules:,}\n\n')
        if not summary_df.empty and combos_with_rules > 0:
            best = summary_df.iloc[0]
            f.write('Best combination (most rules, then highest lift):\n')
            f.write(f'{"-" * 60}\n')
            f.write(f'  Support:         {best["Support"]}\n')
            f.write(f'  Confidence:      {best["Confidence"]}\n')
            f.write(f'  Lift threshold:  {best["Lift_threshold"]}\n')
            f.write(f'  Number of Rules: {int(best["Number_of_Rules"])}\n')
            f.write(f'  Max Lift:        {best["Max_Lift"]}\n')
            f.write(f'  Avg Lift:        {best["Avg_Lift"]}\n')

    print('  > Cleaning up empty folders...')
    removed_conf, removed_sup = cleanup_empty_folders(output_dir)
    print(f'  > Removed {removed_conf} conf folder(s) and {removed_sup} sup folder(s)')

    with open(output_dir / 'exploration_summary.txt', 'a') as f:
        f.write('\nParallelism:\n')
        f.write(f'  n_jobs used      : {n_jobs} '
                f'(of {_CPU_CORES} logical / {_PERF_CORES} perf cores)\n\n')
        f.write('Folder cleanup:\n')
        f.write(f'  conf dirs removed: {removed_conf}\n')
        f.write(f'  sup dirs removed : {removed_sup}\n')

    plot_heatmaps(
        summary_df, output_dir,
        lift_neutral_half_window=lift_neutral_half_window,
        lift_delta=lift_delta,
    )

    print(f'  > summary saved to {output_dir / "summary.csv"}')
    print(f'  > total combinations: {total_combos:,}  |  '
          f'with rules: {combos_with_rules:,}')

    return summary_df


# ---------------------------------------------------------------------------
# Auto-calibration
# ---------------------------------------------------------------------------

def calibrate_parameters(
    encoded_df: pd.DataFrame,
    sup_delta: float       = 0.02,
    lift_delta: float      = 0.05,
    conf_delta: float      = 0.05,
    conf_min_floor: float  = 0.05,
    conf_max: float        = 1.00,
) -> dict | None:
    """
    Auto-calibrate sup_min, sup_max, lift_max, and conf_min from item
    frequencies.  lift_min is always set to 0.0 to include negative
    correlations in the grid from the start.

    Returns None if no 2-itemsets can be formed (transactions too sparse).
    """
    print('  > Calibrating parameters from item frequencies...')

    item_supports = encoded_df.mean().sort_values()
    print('  > Item supports:')
    for item, sup in item_supports.items():
        print(f'    {item}: {sup:.4f}')

    if len(item_supports) < 2:
        print('  > WARNING: fewer than 2 items — cannot form pairwise rules.')
        return None

    rarest = item_supports.iloc[0]
    second = item_supports.iloc[1]
    freq_2 = item_supports.iloc[-2]

    raw_sup_min = rarest * second
    sup_min     = max(
        round(np.floor(raw_sup_min / sup_delta) * sup_delta, 4), sup_delta
    )

    scan_grid                = np.round(
        np.arange(sup_min, freq_2 + sup_delta / 2, sup_delta), 4
    )
    sup_max                  = sup_min
    prev_had_2itemsets       = False
    fi_first_with_2itemsets  = None

    for t in scan_grid:
        fi = fpgrowth(encoded_df, min_support=t, use_colnames=True)
        if fi.empty:
            break
        has_2 = (fi['itemsets'].apply(len).max() >= 2) if not fi.empty else False
        if has_2:
            if fi_first_with_2itemsets is None:
                fi_first_with_2itemsets = fi
            sup_max            = t
            prev_had_2itemsets = True
        elif prev_had_2itemsets:
            break

    raw_lift_max = 1.0 / rarest
    lift_max     = min(round(np.ceil(raw_lift_max * 2) / 2, 1), 10.0)

    if not prev_had_2itemsets:
        print('    - WARNING: no 2-itemsets found at any support threshold.')
        print('      Transactions are too sparse to generate association rules.')
        print('-' * 50)
        return None

    calibrated_conf_min = conf_min_floor
    max_conf            = None
    try:
        if fi_first_with_2itemsets is not None and not fi_first_with_2itemsets.empty:
            rules_probe = association_rules(
                fi_first_with_2itemsets, metric='confidence', min_threshold=0.01
            )
            if not rules_probe.empty:
                max_conf            = rules_probe['confidence'].max()
                calibrated          = round(
                    np.floor(max_conf / conf_delta) * conf_delta, 4
                )
                calibrated_conf_min = max(calibrated, conf_min_floor)
                print(
                    f'  > conf_min calibrated to {calibrated_conf_min} '
                    f'(max observed confidence={max_conf:.4f}, '
                    f'floor={conf_min_floor})'
                )
            else:
                print(
                    f'  > Note: no rules at conf=0.01 for sup_min={sup_min} '
                    f'— conf_min stays at floor={conf_min_floor}'
                )
    except Exception:
        pass

    params = {
        'sup_min':           sup_min,
        'sup_max':           sup_max,
        'sup_delta':         sup_delta,
        'conf_min':          calibrated_conf_min,
        'conf_max':          conf_max,
        'conf_delta':        conf_delta,
        'lift_min':          0.0,   # always 0.0 — negative correlations included
        'lift_max':          lift_max,
        'lift_delta':        lift_delta,
        # Diagnostic values for calibration_log.txt.
        'raw_sup_min':       round(raw_sup_min, 6),
        'raw_lift_max':      round(raw_lift_max, 4),
        'max_conf_observed': round(max_conf, 4) if max_conf is not None else None,
        'item_supports':     item_supports.round(4).to_dict(),
    }

    print(
        f'  > calibrated: sup_min={sup_min} (raw={raw_sup_min:.4f}), '
        f'sup_max={sup_max}, conf_min={calibrated_conf_min} (from data), '
        f'lift_max={lift_max} (raw ceiling={raw_lift_max:.2f})'
    )
    print('-' * 50)

    return params


# ---------------------------------------------------------------------------
# Calibration log writer
# ---------------------------------------------------------------------------

def _write_calibration_log(
    k_dir: Path,
    k: int,
    n_transactions: int,
    item_supports: pd.Series,
    params: dict | None,
    auto_calibrate: bool,
    manual_params: dict = None,
) -> None:
    """
    Write per-k calibration artefacts:

        item_supports.csv    — one row per feature, sorted ascending
        calibration_log.txt  — human-readable parameter summary

    Called for every k, including skipped ones (params=None), so the log
    always documents why a k was included or excluded.
    """
    k_dir = Path(k_dir)
    k_dir.mkdir(parents=True, exist_ok=True)

    sup_df = pd.DataFrame({
        'item':        list(item_supports.index),
        'support_raw': [f'{v:.4f}' for v in item_supports.values],
        'support_pct': [f'{v * 100:.2f}' for v in item_supports.values],
    })
    sup_df.to_csv(k_dir / 'item_supports.csv', index=False)

    with open(k_dir / 'calibration_log.txt', 'w') as f:
        f.write(f'CALIBRATION LOG — k={k}\n')
        f.write(f'{"=" * 60}\n\n')
        f.write(f'Transactions : {n_transactions:,}\n')
        f.write(f'Items        : {len(item_supports)}\n\n')
        f.write(f'Item Supports (sorted ascending):\n{"-" * 40}\n')
        for item, sup in item_supports.items():
            f.write(f'  {item:<12} {sup:.4f}\n')
        f.write('\n')

        if not auto_calibrate:
            f.write('Mode: MANUAL (auto_calibrate=False)\n')
            if manual_params:
                f.write(f'  sup_min  : {manual_params.get("sup_min")}\n')
                f.write(f'  sup_max  : {manual_params.get("sup_max")}\n')
                f.write(f'  conf_min : {manual_params.get("conf_min")}\n')
                f.write(f'  lift_max : {manual_params.get("lift_max")}\n')
            return

        f.write('Mode: AUTO-CALIBRATED\n\n')

        if params is None:
            f.write('Result: SKIPPED\n')
            f.write('Reason: no 2-itemsets found at any support threshold.\n')
            f.write('  Transactions are too sparse to generate association rules.\n')
            f.write('  This typically occurs at low k values where most samples\n')
            f.write('  have only 1 CF neighbour, producing single-item transactions.\n')
            return

        f.write('Calibrated Parameters:\n')
        f.write(f'  sup_min          : {params["sup_min"]}  '
                f'(raw product = {params["raw_sup_min"]:.6f})\n')
        f.write(f'  sup_max          : {params["sup_max"]}\n')
        f.write(f'  sup_delta        : {params["sup_delta"]}\n')
        f.write(f'  conf_min         : {params["conf_min"]}  '
                f'(max observed = {params["max_conf_observed"]})\n')
        f.write(f'  conf_max         : {params["conf_max"]}\n')
        f.write(f'  conf_delta       : {params["conf_delta"]}\n')
        f.write(f'  lift_min         : {params["lift_min"]}  '
                f'(0.0 — negative correlations included)\n')
        f.write(f'  lift_max         : {params["lift_max"]}  '
                f'(raw ceiling = {params["raw_lift_max"]:.4f}, capped at 10.0)\n')
        f.write(f'  lift_delta       : {params["lift_delta"]}\n')


# ---------------------------------------------------------------------------
# K-comparison experiment
# ---------------------------------------------------------------------------

def run_k_comparison(
    k_labels_map: dict,
    output_dir: Path,
    auto_calibrate: bool     = True,
    sup_min: float           = 0.02,
    sup_max: float           = 0.50,
    sup_delta: float         = 0.02,
    conf_min: float          = 0.05,
    conf_max: float          = 1.00,
    conf_delta: float        = 0.05,
    lift_min: float          = 0.0,
    lift_max: float          = 5.0,
    lift_delta: float        = 0.05,
    lift_neutral_half_window: float = 0.25,
) -> dict:
    """
    Run explore_association_rules for each k and produce a cross-k comparison.

    lift_min=0.0 (default) ensures negative correlations are included in the
    grid for every k in both auto and manual mode.

    Parameters
    ----------
    k_labels_map    : dict mapping k (int) → Path of the labels CSV.
    output_dir      : root output directory; per-k and comparison sub-folders
                      are created automatically.
    auto_calibrate  : if True, calibrate sup_min/sup_max/lift_max/conf_min
                      from item frequencies for each k.
    *               : remaining grid parameters used as fallback (manual mode)
                      or as conf_min_floor / conf_max / deltas (auto mode).

    Returns
    -------
    dict mapping k → summary DataFrame for that k.
    """
    output_dir = Path(output_dir)
    comp_dir   = output_dir / 'k_comparison'
    comp_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n{"=" * 70}')
    print(f'K-VARIATION EXPERIMENT — {len(k_labels_map)} k values')
    print(f'{"=" * 70}')
    print(f'  > k values        : {sorted(k_labels_map.keys())}')
    print(f'  > auto_calibrate  : {auto_calibrate}')
    print('-' * 50)

    k_summaries      = {}
    comparison_rows  = []

    for k in sorted(k_labels_map.keys()):
        labels_csv = Path(k_labels_map[k])
        k_dir      = output_dir / f'k_{k}'
        k_dir.mkdir(parents=True, exist_ok=True)

        print(f'\n{"=" * 70}')
        print(f'  k = {k}  |  {labels_csv.name}')
        print(f'{"=" * 70}')

        if not labels_csv.exists():
            print(f'  > File not found: {labels_csv}')
            comparison_rows.append({
                'k': k, 'skipped_reason': 'file_not_found',
                'n_transactions': None, 'n_items': None,
                'rarest_item_support': None, 'sup_min_used': None,
                'sup_max_used': None, 'lift_max_used': None,
                'conf_min_used': None, 'summary_rows': 0,
                'combos_with_rules': 0, 'max_rules_any_combo': 0,
                'avg_lift_best_combo': None, 'max_lift_observed': None,
            })
            continue

        df_encoded    = extract_labels(labels_csv)
        item_supports = df_encoded.mean().sort_values()

        if auto_calibrate:
            params = calibrate_parameters(
                encoded_df     = df_encoded,
                sup_delta      = sup_delta,
                lift_delta     = lift_delta,
                conf_delta     = conf_delta,
                conf_min_floor = conf_min,
                conf_max       = conf_max,
            )
            _write_calibration_log(
                k_dir=k_dir, k=k,
                n_transactions=len(df_encoded),
                item_supports=item_supports,
                params=params,
                auto_calibrate=True,
            )

            if params is None:
                print(f'  > Skipping k={k} — insufficient co-occurrences for rules.')
                comparison_rows.append({
                    'k': k, 'skipped_reason': 'too_sparse',
                    'n_transactions': len(df_encoded),
                    'n_items': df_encoded.shape[1],
                    'rarest_item_support': round(item_supports.iloc[0], 4),
                    'sup_min_used': None, 'sup_max_used': None,
                    'lift_max_used': None, 'conf_min_used': None,
                    'summary_rows': 0, 'combos_with_rules': 0,
                    'max_rules_any_combo': 0, 'avg_lift_best_combo': None,
                    'max_lift_observed': None,
                })
                continue

            k_sup_min  = params['sup_min']
            k_sup_max  = params['sup_max']
            k_lift_max = params['lift_max']
            k_conf_min = params['conf_min']
            k_lift_min = params['lift_min']   # always 0.0

        else:
            k_sup_min  = sup_min
            k_sup_max  = sup_max
            k_lift_max = lift_max
            k_conf_min = conf_min
            k_lift_min = lift_min

            _write_calibration_log(
                k_dir=k_dir, k=k,
                n_transactions=len(df_encoded),
                item_supports=item_supports,
                params=None,
                auto_calibrate=False,
                manual_params={
                    'sup_min': k_sup_min, 'sup_max': k_sup_max,
                    'conf_min': k_conf_min, 'lift_max': k_lift_max,
                },
            )

        summary_df = explore_association_rules(
            df          = df_encoded,
            output_dir  = k_dir,
            sup_min     = k_sup_min,  sup_max  = k_sup_max,  sup_delta  = sup_delta,
            conf_min    = k_conf_min, conf_max = conf_max,   conf_delta = conf_delta,
            lift_min    = k_lift_min, lift_max = k_lift_max, lift_delta = lift_delta,
            lift_neutral_half_window = lift_neutral_half_window,
        )

        k_summaries[k] = summary_df

        has_rules_col = (
            not summary_df.empty and 'Number_of_Rules' in summary_df.columns
        )
        with_rules = (
            summary_df[summary_df['Number_of_Rules'] > 0]
            if has_rules_col
            else pd.DataFrame()
        )
        comparison_rows.append({
            'k':                   k,
            'skipped_reason':      '',
            'n_transactions':      len(df_encoded),
            'n_items':             df_encoded.shape[1],
            'rarest_item_support': round(item_supports.iloc[0], 4),
            'sup_min_used':        k_sup_min,
            'sup_max_used':        k_sup_max,
            'lift_max_used':       k_lift_max,
            'conf_min_used':       k_conf_min,
            'summary_rows':        len(summary_df),
            'combos_with_rules':   len(with_rules),
            'max_rules_any_combo': int(summary_df['Number_of_Rules'].max()) if has_rules_col else 0,
            'avg_lift_best_combo': round(with_rules['Avg_Lift'].max(), 4) if not with_rules.empty else 0.0,
            'max_lift_observed':   round(summary_df['Max_Lift'].max(), 4) if has_rules_col else 0.0,
        })

    print(f'\n{"=" * 70}')
    print('  > Building cross-k comparison...')

    comp_df = pd.DataFrame(comparison_rows)
    comp_df.to_csv(comp_dir / 'k_comparison_summary.csv', index=False)

    with open(comp_dir / 'k_comparison_summary.txt', 'w') as f:
        f.write('K-VARIATION EXPERIMENT SUMMARY\n')
        f.write(f'{"=" * 70}\n\n')
        f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
        f.write(f'k values tested : {sorted(k_labels_map.keys())}\n')
        f.write(f'auto_calibrate  : {auto_calibrate}\n\n')

        skipped = comp_df[comp_df['skipped_reason'] != ''] if not comp_df.empty else pd.DataFrame()
        ran     = comp_df[comp_df['skipped_reason'] == ''] if not comp_df.empty else pd.DataFrame()

        if not skipped.empty:
            f.write(f'Skipped k values ({len(skipped)}):\n{"-" * 40}\n')
            for _, row in skipped.iterrows():
                reason = row['skipped_reason']
                if reason == 'too_sparse':
                    detail = (
                        f'no 2-itemsets found '
                        f'(n_transactions={int(row["n_transactions"]):,}, '
                        f'rarest_support={row["rarest_item_support"]})'
                    )
                elif reason == 'file_not_found':
                    detail = 'input CSV not found'
                else:
                    detail = reason
                f.write(f'  k={int(row["k"])}: {detail}\n')
            f.write('\n')

        f.write(f'Results per k (processed only):\n{"-" * 60}\n')
        if not ran.empty:
            f.write(ran.drop(columns='skipped_reason').to_string(index=False))
        else:
            f.write('  No k values produced results.\n')
        f.write('\n\n')

        if not ran.empty and ran['max_rules_any_combo'].max() > 0:
            best_k = ran.loc[ran['max_rules_any_combo'].idxmax(), 'k']
            f.write(
                f'Most rules: k={best_k} '
                f'({ran["max_rules_any_combo"].max()} rules at best combination)\n'
            )
        else:
            f.write('No rules found for any k value.\n')

    print('  > Saved k_comparison_summary.csv and .txt')

    # Cross-k heatmaps.
    all_summaries = []
    for k, sdf in k_summaries.items():
        tmp      = sdf.copy()
        tmp['k'] = k
        all_summaries.append(tmp)

    if not all_summaries:
        print('  > No rules found in any k — skipping cross-k heatmaps.')
        print(f'  > Cross-k comparison saved to {comp_dir}/')
        return k_summaries

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=FutureWarning)
        combined = pd.concat(all_summaries, ignore_index=True)

    if combined.empty or 'Lift_threshold' not in combined.columns:
        print('  > No rules found in any k — skipping cross-k heatmaps.')
        print(f'  > Cross-k comparison saved to {comp_dir}/')
        return k_summaries

    combined['Lift_display'] = (
        (combined['Lift_threshold'] / lift_delta).round() * lift_delta
    ).round(4)

    neutral_lo, neutral_hi = _neutral_window(lift_neutral_half_window)

    for x_col, suffix in [
        ('Support',      'k_support'),
        ('Confidence',   'k_confidence'),
        ('Lift_display', 'k_lift'),
    ]:
        x_label   = 'Lift' if 'lift' in suffix else x_col
        is_lift   = 'lift' in suffix

        pivot = (
            combined.groupby(['k', x_col])['Number_of_Rules']
            .max()
            .unstack(level=x_col)
            .sort_index(ascending=False)
            .fillna(0)
            .astype(int)
        )

        if is_lift:
            pivot = pivot.loc[
                :, ~pivot.columns.to_series().between(
                    neutral_lo, neutral_hi, inclusive='both'
                )
            ]
            if (pivot != 0).any(axis=0).any():
                last_nz = int(np.where((pivot != 0).any(axis=0).values)[0].max())
                pivot   = pivot.iloc[:, :last_nz + 1]

        n_cols = len(pivot.columns)
        n_rows = len(pivot.index)
        fig, ax = plt.subplots(
            figsize=(max(10, n_cols * 0.75), max(4, n_rows * 0.6))
        )

        img = ax.imshow(
            pivot.values, aspect='auto', cmap='YlOrBr', interpolation='nearest'
        )

        ax.set_xticks(range(n_cols))
        ax.set_xticklabels(
            [f'{v:.2f}' if isinstance(v, float) else str(v) for v in pivot.columns],
            rotation=40, ha='right', fontsize=8,
        )
        ax.set_yticks(range(n_rows))
        ax.set_yticklabels([f'k={v}' for v in pivot.index], fontsize=9)

        ax.set_xlabel(x_label, fontsize=11, labelpad=8)
        ax.set_ylabel('k',     fontsize=11, labelpad=8)
        ax.set_title(
            f'Max Number of Rules — k vs {x_label}\n'
            f'(darker = more rules; max over the other two parameters)',
            fontsize=11, pad=14,
        )

        max_val = pivot.values.max() if pivot.values.max() > 0 else 1
        for ri in range(n_rows):
            for ci in range(n_cols):
                val = pivot.values[ri, ci]
                if val > 0:
                    txt_color = 'white' if (val / max_val) > 0.55 else 'black'
                    ax.text(
                        ci, ri, str(val),
                        ha='center', va='center', fontsize=7, color=txt_color,
                    )

        cbar = plt.colorbar(img, ax=ax, fraction=0.025, pad=0.02)
        cbar.set_label('Number of Rules', fontsize=9)
        plt.tight_layout()

        fig.savefig(
            comp_dir / f'heatmap_{suffix}.png', dpi=150, bbox_inches='tight'
        )
        plt.close(fig)
        print(f'    > saved k_comparison/heatmap_{suffix}.png')

    print(f'  > Cross-k comparison saved to {comp_dir}/')
    return k_summaries


# ---------------------------------------------------------------------------
# Experiment labelling
# ---------------------------------------------------------------------------

def _experiment_label(
    auto_calibrate: bool,
    sup_min, sup_max, sup_delta,
    conf_min, conf_max, conf_delta,
    lift_min, lift_max, lift_delta,
    lift_neutral_half_window: float,
) -> str:
    """
    Build a human-readable folder name for one experiment configuration.

    conf_min shown in the label is the floor value, not the per-k calibrated
    value — it identifies the configuration, not derived per-k parameters.
    """
    def fmt(v):
        return f'{v:.2f}'

    if auto_calibrate:
        prefix    = 'auto'
        sup_part  = f'sup=auto_d{fmt(sup_delta)}'
        lift_part = f'lift=auto_d{fmt(lift_delta)}_w{fmt(lift_neutral_half_window)}'
    else:
        prefix    = 'manual'
        sup_part  = f'sup={fmt(sup_min)}-{fmt(sup_max)}_d{fmt(sup_delta)}'
        lift_part = (
            f'lift={fmt(lift_min)}-{fmt(lift_max)}'
            f'_d{fmt(lift_delta)}_w{fmt(lift_neutral_half_window)}'
        )

    conf_part = f'conf={fmt(conf_min)}-{fmt(conf_max)}_d{fmt(conf_delta)}'
    return f'{prefix}_{sup_part}_{conf_part}_{lift_part}'


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def main(
    regions: list                    = None,
    k_values: list[int]              = None,
    auto_calibrate: bool             = True,
    sup_min: float                   = 0.02,
    sup_max: float                   = 0.50,
    sup_delta: float                 = 0.02,
    conf_min: float                  = 0.05,
    conf_max: float                  = 1.00,
    conf_delta: float                = 0.05,
    lift_min: float                  = 0.0,
    lift_max: float                  = 5.0,
    lift_delta: float                = 0.05,
    lift_neutral_half_window: float  = 0.25,
    base_dir: Path                   = None,
) -> None:
    """
    Run association-rule mining for all regions and k values.

    Parameters
    ----------
    regions                  : list of region names to process.
    k_values                 : neighbourhood sizes produced by feature_importance.py.
    auto_calibrate           : if True, calibrate grid bounds from item frequencies.
    sup_min / sup_max        : support grid bounds (fallback when auto_calibrate=False).
    sup_delta                : support grid step size.
    conf_min / conf_max      : confidence grid bounds; conf_min is the floor when
                               auto_calibrate=True.
    conf_delta               : confidence grid step size.
    lift_min / lift_max      : lift grid bounds; lift_min=0.0 includes negative
                               correlations.
    lift_delta               : lift grid step size.
    lift_neutral_half_window : half-width of the neutral lift window to exclude.
    base_dir                 : project root directory.
    """
    print(
        f'  > Parallel backend — {_CPU_CORES} logical cores / '
        f'{_PERF_CORES} perf cores (joblib loky)'
    )

    if base_dir is None:
        if Path('/kaggle/working').exists():
            base_dir = Path('/kaggle/working')
        elif Path('/content').exists():
            base_dir = Path('/content')
        else:
            base_dir = Path(__file__).resolve().parent.parent
    base_dir = Path(base_dir)

    if regions is None:
        regions = ['northeast', 'south']
    if k_values is None:
        k_values = [1, 3, 5, 7]

    results_dir = base_dir / 'results'

    exp_label = _experiment_label(
        auto_calibrate=auto_calibrate,
        sup_min=sup_min,   sup_max=sup_max,   sup_delta=sup_delta,
        conf_min=conf_min, conf_max=conf_max, conf_delta=conf_delta,
        lift_min=lift_min, lift_max=lift_max, lift_delta=lift_delta,
        lift_neutral_half_window=lift_neutral_half_window,
    )

    for region in regions:
        important_features_dir = results_dir / region / 'important_features'
        ar_output_dir          = results_dir / region / 'association_rules' / exp_label
        ar_output_dir.mkdir(parents=True, exist_ok=True)

        print('\n' + '=' * 70)
        print(f'ASSOCIATION RULES — {region.upper()}')
        print(f'Experiment: {exp_label}')
        print('=' * 70 + '\n')

        # Prefer aggregated format; fall back to per-(sample, CF) format.
        k_labels_map = {}
        for k in k_values:
            p_agg  = important_features_dir / f'k_{k}' / 'aggregated_labels_by_sample.csv'
            p_orig = important_features_dir / f'k_{k}' / 'labels_only_unique.csv'
            if p_agg.exists():
                k_labels_map[k] = (p_agg, 'aggregated (preferred)')
            elif p_orig.exists():
                k_labels_map[k] = (p_orig, 'original (fallback)')

        ks_with_agg  = sum(1 for _, (_, src) in k_labels_map.items() if 'aggregated' in src)
        ks_with_orig = len(k_labels_map) - ks_with_agg

        log_path = ar_output_dir / 'experiment_log.txt'
        with open(log_path, 'w') as f:
            f.write('EXPERIMENT LOG\n')
            f.write(f'{"=" * 70}\n\n')
            f.write(f'Generated     : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
            f.write(f'Region        : {region.upper()}\n')
            f.write(f'Experiment    : {exp_label}\n')
            f.write(f'Output dir    : {ar_output_dir}\n\n')
            f.write('Configuration:\n')
            f.write(f'  auto_calibrate          : {auto_calibrate}\n')
            f.write(f'  sup_min / sup_max       : {sup_min} / {sup_max}  '
                    f'(fallback when auto_calibrate=False)\n')
            f.write(f'  sup_delta               : {sup_delta}\n')
            f.write(f'  conf_min / conf_max     : {conf_min} / {conf_max}  '
                    f'(conf_min is floor when calibrating)\n')
            f.write(f'  conf_delta              : {conf_delta}\n')
            f.write(f'  lift_min / lift_max     : {lift_min} / {lift_max}  '
                    f'(lift_min=0.0 includes negative correlations)\n')
            f.write(f'  lift_delta              : {lift_delta}\n')
            f.write(
                f'  lift_neutral_half_window: {lift_neutral_half_window}  '
                f'(excludes [{round(1.0 - lift_neutral_half_window, 4)}, '
                f'{round(1.0 + lift_neutral_half_window, 4)}])\n\n'
            )
            f.write(f'Parallelism   : {_CPU_CORES} logical / '
                    f'{_PERF_CORES} perf cores (joblib loky)\n\n')

            if not k_labels_map:
                f.write('Input files   : NONE FOUND\n')
                f.write(f'  Searched under: {important_features_dir}\n')
                f.write('  Run feature_importance.py first.\n')
            else:
                f.write(f'Input files found ({len(k_labels_map)} k values):\n')
                f.write(f'{"-" * 60}\n')
                for k_val, (path, src) in sorted(k_labels_map.items()):
                    f.write(f'  k={k_val:<3}  [{src}]  {path}\n')
                f.write(f'\n  {ks_with_agg} aggregated (preferred)\n')
                if ks_with_orig:
                    f.write(f'  {ks_with_orig} original (fallback)\n')

        if not k_labels_map:
            print(f'  > No labels files found under {important_features_dir}')
            print('    Run feature_importance.py first.')
            continue

        print(f'  > Labels found for k = {sorted(k_labels_map.keys())}')
        print(f'    - {ks_with_agg} k values using aggregated format (preferred)')
        if ks_with_orig:
            print(f'    - {ks_with_orig} k values using original format (fallback)')

        k_paths_map = {k: path for k, (path, _) in k_labels_map.items()}

        k_summaries = run_k_comparison(
            k_labels_map             = k_paths_map,
            output_dir               = ar_output_dir,
            auto_calibrate           = auto_calibrate,
            sup_min                  = sup_min,  sup_max  = sup_max,  sup_delta  = sup_delta,
            conf_min                 = conf_min, conf_max = conf_max, conf_delta = conf_delta,
            lift_min                 = lift_min, lift_max = lift_max, lift_delta = lift_delta,
            lift_neutral_half_window = lift_neutral_half_window,
        )

        k_max_per_k = {
            k: int(sdf['Number_of_Rules'].max())
            for k, sdf in k_summaries.items()
            if not sdf.empty and 'Number_of_Rules' in sdf.columns
        }
        sum_rules = sum(k_max_per_k.values())
        max_rules = max(k_max_per_k.values()) if k_max_per_k else 0
        best_k    = max(k_max_per_k, key=k_max_per_k.get) if k_max_per_k else None
        k_ran     = sorted(k_summaries.keys())
        k_skipped = sorted(set(k_labels_map.keys()) - set(k_summaries.keys()))
        k_with_rules = sorted(k for k, n in k_max_per_k.items() if n > 0)

        with open(log_path, 'a') as f:
            f.write(f'\n{"=" * 70}\n')
            f.write('Post-run Summary:\n')
            f.write(f'{"-" * 60}\n')
            f.write(f'  k values ran              : {k_ran}\n')
            f.write(f'  k values skipped          : '
                    f'{k_skipped if k_skipped else "none"}\n')
            f.write(f'  k values with rules       : '
                    f'{k_with_rules if k_with_rules else "none"}\n')
            f.write(f'  Sum of max rules across k : {sum_rules}  '
                    f'(total signal across all k)\n')
            f.write(f'  Max rules in best combo   : {max_rules}  '
                    f'(strongest single combination, k={best_k})\n')
            f.write(f'  Completed at              : '
                    f'{datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')

    print('\n' + '=' * 70)
    print('Done.')
    print('=' * 70 + '\n')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile src/main.py
"""
main.py — Pipeline orchestrator
================================
Runs the three pipeline modules in sequence or individually.
"""

import argparse
import importlib.util
import sys
import time
import traceback
from pathlib import Path

# ---------------------------------------------------------------------------
# Source file locations
# ---------------------------------------------------------------------------
_HERE = Path(__file__).resolve().parent

_SRC = {
    1: _HERE / 'create_dataset.py',
    2: _HERE / 'feature_importance.py',
    3: _HERE / 'macroscopic_experiment_association_rules.py',
}

_STEP_NAMES = {
    1: 'Create Dataset',
    2: 'Feature Importance (CategoricalBoCSoR)',
    3: 'Association Rules (FP-Growth)',
}

# ---------------------------------------------------------------------------
# Utility helpers
# ---------------------------------------------------------------------------
def _banner(msg: str, char: str = '=', width: int = 70) -> None:
    print(f'\n{char * width}')
    print(f'  {msg}')
    print(f'{char * width}')

def _load_module(path: Path, name: str):
    spec   = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def _check_source_files() -> bool:
    ok = True
    for step, path in _SRC.items():
        if not path.exists():
            print(f'  [ERROR] Source for Step {step} not found: {path}')
            ok = False
    return ok

# ---------------------------------------------------------------------------
# Output pre-existence checks
# ---------------------------------------------------------------------------
def _outputs_step1_exist(survey_year: str, output_dir: str, regions: list[str]) -> bool:
    for reg in regions:
        p = Path(output_dir) / f'acs_income_{reg}_{survey_year}.csv'
        if not p.exists():
            return False
    return True

def _outputs_step2_exist(regions: list[str], k_values: list[int]) -> bool:
    results_dir = _HERE.parent / 'results'
    for region in regions:
        for k in k_values:
            p = results_dir / region / 'important_features' / f'k_{k}' / 'transactions_values.csv'
            if not p.exists():
                return False
    return True

def _inputs_step2_exist(regions: list[str], survey_year: str, output_dir: str) -> bool:
    data_dir = Path(output_dir)
    ok = True
    for region in regions:
        p = data_dir / f'acs_income_{region}_{survey_year}.csv'
        if not p.exists():
            print(f'  [WARNING] Missing input for Step 2: {p}')
            ok = False
    return ok

def _inputs_step3_exist(regions: list[str], k_values: list[int]) -> bool:
    results_dir = _HERE.parent / 'results'
    ok = True
    for region in regions:
        for k in k_values:
            base  = results_dir / region / 'important_features' / f'k_{k}'
            found = (base / 'aggregated_labels_by_sample.csv').exists() or (base / 'labels_only_unique.csv').exists()
            if not found:
                print(f'  [WARNING] Missing input for Step 3: {base}/ (aggregated_labels_by_sample.csv or labels_only_unique.csv)')
                ok = False
    return ok

# ---------------------------------------------------------------------------
# Step runners
# ---------------------------------------------------------------------------
def run_step1(args: argparse.Namespace) -> bool:
    _banner(f'STEP 1 — {_STEP_NAMES[1]}')
    if not args.force and _outputs_step1_exist(args.survey_year, args.output_dir, args.regions):
        print(f'  > Output files already present (survey_year={args.survey_year}). Use --force to overwrite.')
        return True

    mod = _load_module(_SRC[1], 'create_dataset')
    t0 = time.perf_counter()
    try:
        mod.main(
            survey_year                = args.survey_year,
            horizon                    = args.horizon,
            random_seed                = args.random_seed,
            output_dir                 = args.output_dir,
            income_threshold_northeast = args.income_threshold_ne,
            income_threshold_south     = args.income_threshold_south,
            income_threshold_usa       = args.income_threshold_usa,
            regions_to_build           = args.regions,
        )
    except Exception:
        print('\n  [ERROR] Step 1 failed:')
        traceback.print_exc()
        return False
    print(f'\n  > Step 1 completed in {time.perf_counter() - t0:.1f}s')
    return True

def run_step2(args: argparse.Namespace) -> bool:
    _banner(f'STEP 2 — {_STEP_NAMES[2]}')
    if not _inputs_step2_exist(args.regions, args.survey_year, args.output_dir):
        print('  [ERROR] Required inputs are missing — run Step 1 first.')
        return False
    if not args.force and _outputs_step2_exist(args.regions, args.k_values):
        print('  > Output files already present for all regions and k values. Use --force to overwrite.')
        return True

    mod = _load_module(_SRC[2], 'feature_importance')
    data_dir = Path(args.output_dir)
    regions  = {r: data_dir / f'acs_income_{r}_{args.survey_year}.csv' for r in args.regions}

    t0 = time.perf_counter()
    try:
        mod.main(
            survey_year    = args.survey_year,
            regions        = regions,
            k_values       = args.k_values,
            perc_threshold = args.perc_threshold,
            target_col     = args.target_col,
            base_dir       = _HERE.parent,
        )
    except Exception:
        print('\n  [ERROR] Step 2 failed:')
        traceback.print_exc()
        return False
    print(f'\n  > Step 2 completed in {time.perf_counter() - t0:.1f}s')
    return True

def run_step3(args: argparse.Namespace) -> bool:
    _banner(f'STEP 3 — {_STEP_NAMES[3]}')
    if not _inputs_step3_exist(args.regions, args.k_values):
        print('  [ERROR] Required inputs are missing — run Step 2 first.')
        return False

    mod = _load_module(_SRC[3], 'macroscopic_experiment_association_rules')
    t0 = time.perf_counter()
    try:
        mod.main(
            regions                  = args.regions,
            k_values                 = args.k_values,
            auto_calibrate           = args.auto_calibrate,
            sup_min                  = args.sup_min,
            sup_max                  = args.sup_max,
            sup_delta                = args.sup_delta,
            conf_min                 = args.conf_min,
            conf_max                 = args.conf_max,
            conf_delta               = args.conf_delta,
            lift_min                 = args.lift_min,
            lift_max                 = args.lift_max,
            lift_delta               = args.lift_delta,
            lift_neutral_half_window = args.lift_neutral_half_window,
            base_dir                 = _HERE.parent,
        )
    except Exception:
        print('\n  [ERROR] Step 3 failed:')
        traceback.print_exc()
        return False
    print(f'\n  > Step 3 completed in {time.perf_counter() - t0:.1f}s')
    return True

# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        prog            = 'main.py',
        description     = 'Pipeline orchestrator: create_dataset → feature_importance → association_rules',
        formatter_class = argparse.RawDescriptionHelpFormatter,
    )
    parser.add_argument('--steps', nargs='+', type=int, choices=[1, 2, 3], default=[1, 2, 3], metavar='{1,2,3}')
    parser.add_argument('--force', action='store_true', help='Force overwrite of existing output files.')
    parser.add_argument('--dry-run', action='store_true', help='Print execution plan without running.')

    g1 = parser.add_argument_group('Step 1: dataset')
    g1.add_argument('--survey-year', default='2024')
    g1.add_argument('--horizon', default='1-Year')
    g1.add_argument('--random-seed', type=int, default=42)
    g1.add_argument('--output-dir', default='data')
    g1.add_argument('--income-threshold-ne', type=int, default=110_000)
    g1.add_argument('--income-threshold-south', type=int, default=90_000)
    g1.add_argument('--income-threshold-usa', type=int, default=100_000, help='USA global income threshold in USD.')

    g2 = parser.add_argument_group('Step 2: feature importance')
    g2.add_argument('--k-values', nargs='+', type=int, default=[1, 3, 5, 7])
    g2.add_argument('--perc-threshold', type=int, default=10)
    g2.add_argument('--target-col', default='INCOME_ABOVE_THRESHOLD')

    parser.add_argument(
        '--regions', nargs='+', choices=['northeast', 'south', 'usa'],
        default=['northeast', 'south'], help='Regions to process (default: northeast south).'
    )

    g3  = parser.add_argument_group('Step 3: association rules')
    cal = g3.add_mutually_exclusive_group()
    cal.add_argument('--auto-calibrate', dest='auto_calibrate', action='store_true', default=True)
    cal.add_argument('--no-auto-calibrate', dest='auto_calibrate', action='store_false')
    g3.add_argument('--sup-min', type=float, default=0.02)
    g3.add_argument('--sup-max', type=float, default=0.50)
    g3.add_argument('--sup-delta', type=float, default=0.02)
    g3.add_argument('--conf-min', type=float, default=0.05)
    g3.add_argument('--conf-max', type=float, default=1.00)
    g3.add_argument('--conf-delta', type=float, default=0.05)
    g3.add_argument('--lift-min', type=float, default=0.0)
    g3.add_argument('--lift-max', type=float, default=5.0)
    g3.add_argument('--lift-delta', type=float, default=0.05)
    g3.add_argument('--lift-neutral-half-window', type=float, default=0.25)

    return parser.parse_args()

# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
def main() -> None:
    args = _parse_args()
    _banner('PIPELINE ORCHESTRATOR')
    print(f'  Steps selected               : {sorted(args.steps)}')
    print(f'  Regions                      : {args.regions}')
    print(f'  Income threshold — Northeast : ${args.income_threshold_ne:,}')
    print(f'  Income threshold — South     : ${args.income_threshold_south:,}')
    print(f'  Income threshold — USA       : ${args.income_threshold_usa:,}')

    if not _check_source_files():
        print('\n  [ERROR] One or more source files are missing. Aborting.')
        sys.exit(1)

    if args.dry_run:
        _banner('DRY RUN — no commands will be executed', char='-')
        for step in sorted(args.steps):
            print(f'  [Step {step}] {_STEP_NAMES[step]}  →  {_SRC[step].name}')
        print()
        return

    _RUNNERS = {1: run_step1, 2: run_step2, 3: run_step3}
    total_t0 = time.perf_counter()
    results = {}

    for step in sorted(args.steps):
        results[step] = _RUNNERS[step](args)
        if not results[step]:
            print(f'\n  [ERROR] Step {step} did not complete successfully. Pipeline interrupted.')
            break

    elapsed = time.perf_counter() - total_t0
    _banner('SUMMARY')
    for step, ok in sorted(results.items()):
        status = '✓  OK' if ok else '✗  FAILED'
        print(f'  Step {step} ({_STEP_NAMES[step]}): {status}')
    print(f'\n  Total elapsed time: {elapsed:.1f}s')
    if not all(results.values()):
        sys.exit(1)

if __name__ == '__main__':
    main()


## Run pipeline

Modify the command below to choose regions and thresholds.


In [ ]:
# Regions: northeast, south, usa (or any combination)
# Adjust income thresholds as needed
!python src/main.py --regions usa --income-threshold-usa 100000


In [ ]:
# Compress results for easy download
!zip -r /kaggle/working/results.zip results/
print("Results zipped.")
